# From messy files to AI-ready data
### A step-by-step walkthrough of every transformation, for people new to data engineering

This notebook takes the raw files of a fictional grocery chain, **FreshCart**, and turns them, one table at a time,
into clean tables that an AI assistant (Databricks Genie) can answer business questions from **correctly**.

You don't need to know anything about data engineering. New ideas are explained in **💡 Concept** boxes at the moment
you first need them, and every table is explained with the **same layout**, so once you've read one you know how to
read them all.

## 0 · How to read this notebook

**Every table section looks like this:**

| Block | What it tells you |
|---|---|
| 🎯 **Goal** + card | In one sentence: why this table exists, what goes in, what comes out, what one row means |
| ⚠️ **What's wrong with the input** | The raw rows, and the problems in them |
| 🗺️ **Column map** | For every output column: which raw column it comes from and what rule changes it |
| 🔧 **Transformation 1, 2, 3...** | Each change **on its own**: *what* we do, *why* (what goes wrong if we skip it), and a **before → after** table showing only the columns involved |
| ▶️ **Build the table** | Runs the project's real SQL file and shows a few finished rows |
| ✅ **Check** | Automatic tests that prove the step worked (each prints ✅ or ❌) |
| 📌 **Recap** | The step in three bullet points |

In the before → after tables, columns named **`before · X`** are the raw value and **`after · Y`** is the clean value.

**Running it:** from the repository root run `pip install -r notebooks/requirements.txt`, open this notebook in
Jupyter and choose *Kernel → Restart & Run All*. It takes about 30 seconds. All numbers are computed by the cells.

---
# 1 · The big picture

### The problem
FreshCart has 9 stores in the US and Canada, an online shop, a loyalty card and weekly promotions. Its data comes
from **six different systems**, each built for its own job, not for analysis:

| System | Sends us | Example of its "own way" of doing things |
|---|---|---|
| Store tills (POS) | every receipt line | store `01042`, date `20260914`, returns as **positive** amounts |
| Website | every online order (JSON) | store `9001` (no zeros), lines **nested** inside the order |
| ERP | products, categories, costs | each product listed **twice** (US and Canada copies) |
| Store master | store details | codes (`NM`, `02`), US dates `05/14/2016` |
| CRM | loyalty members | card numbers, emails, birthdays (**personal data**) |
| Finance | fiscal calendar, exchange rates | UK dates `31/01/2026`, rates on weekdays only |

If a business user asks *"what were net sales by region last week?"* and an AI writes SQL straight on these files, it
will be wrong: it won't match `01042` with `1042`, it will count returns as sales, double every product, and not know
what "last week" means in a retail calendar. **Data engineering fixes all of that once, in a pipeline, so every
answer afterwards is right.**

### The solution: layers
Data flows through layers. Each layer has **one job**, so you always know where something was fixed.

```
   RAW FILES  ───►  BRONZE  ───────────►  SILVER  ─────────────────►  GOLD  ─────────────────►  SEMANTIC
   data/raw/        copy as-is            clean & fix                 organise for business       define KPIs once
   (never edited)   + where-it-came-from  (types, ids, codes,         (facts + dimensions,        (net sales, margin %,
                    labels                 duplicates, personal data)  business rules)             stock on hand)
```

| Layer | Its one job | Golden rule |
|---|---|---|
| **Bronze** | Get every file into a table **without changing anything** | Store everything as text; record which file and row each record came from |
| **Silver** | **Clean** the data | Fix types, ids, codes, duplicates. Keep every *real* event (even voided lines), but flag it. Put broken rows aside with a reason |
| **Gold** | **Organise** the data the way the business thinks | Only what counts as a sale; tables of *events* (facts) and *descriptions* (dimensions); every key matches |
| **Semantic** | **Define** each KPI once | "Net sales", "margin %", "stock" calculated one way, everywhere |

### The roadmap: 16 tables we will build, in this order

The order matters: a step can only use tables built before it (sales need exchange rates, exchange rates need the calendar).

| # | Table we build | From | In one sentence |
|---|---|---|---|
| B | 15 bronze tables | raw files | Copy every file into a table, unchanged |
| S1 | `silver.fiscal_calendar` | bronze calendar | Turn Finance's text calendar into real dates and numbers |
| S2 | `silver.fx_rate_daily` | bronze rates + S1 | A CAD→USD rate for **every** day, including weekends |
| S3 | `silver.store_history` | bronze stores + 3 lookups | Decoded stores, keeping a row per version when a store changes |
| S4 | `silver.product`, `silver.product_cost` | bronze ERP | One row per product (not two), decoded categories, cost by date |
| S5 | `silver.promotion`, `silver.promotion_item` | bronze promos | Decoded promotions |
| S6 | `silver.loyalty_member` (+ restricted) | bronze CRM | Members without personal data; personal data locked away |
| S7 | `silver.pos_sales_line` (+ quarantine) | bronze tills + S2 | Clean till lines: 8 fixes, bad rows set aside |
| S8 | `silver.online_sales_line` | bronze web + S2 | Online order lines in exactly the same shape as till lines |
| S9 | `silver.inventory_daily` | bronze stock | Clean nightly stock counts |
| G1 | `gold.dim_date` | S1 | Calendar with ready-made "last week", "YTD" flags |
| G2 | `gold.dim_store` | S3 | One row per store, with the like-for-like rule applied |
| G3 | `gold.dim_product` | S4 | Business-friendly product list |
| G4 | `gold.dim_promotion` | S5 | Promotion list, plus "no promotion" |
| G5 | `gold.dim_customer` | S6 + sales | Customers, plus "anonymous" and not-yet-known members |
| G6 | `gold.fct_sales_line` | S7 + S8 + dims | **The sales table**: only real sales, with net sales, cost and margin |
| G7 | `gold.fct_inventory_daily` | S9 + dims | Daily stock, valued, with an out-of-stock flag |

---
# 2 · SQL in 10 minutes (on a toy table)

Every transformation is written in **SQL**, the language of databases. You only need **six ideas**. We practise them on
a tiny made-up table of 5 sales lines and 2 stores, which already contains three typical problems:
store ids with and without leading zeros, a **duplicated** line, and a store that isn't in the store list.

In [1]:
import sqlite3
import pandas as pd

toy = sqlite3.connect(":memory:")
toy.executescript('''
CREATE TABLE sales  (receipt TEXT, line INTEGER, store TEXT, amount REAL, margin REAL);
INSERT INTO sales VALUES ('R1', 1, '01042', 6.98, 1.00), ('R1', 2, '01042', 3.99, 2.00),
                         ('R2', 1, '01107', 3.49, 1.40), ('R2', 1, '01107', 3.49, 1.40),   -- duplicate!
                         ('R3', 1, '09999', 5.00, 0.50);                                  -- unknown store!
CREATE TABLE stores (store TEXT PRIMARY KEY, name TEXT);
INSERT INTO stores VALUES ('1042', 'Boston'), ('1107', 'Hartford');
''')
tq = lambda sql: pd.read_sql_query(sql, toy)
display(tq("SELECT * FROM sales"))
display(tq("SELECT * FROM stores"))

,receipt,line,store,amount,margin
0,R1,1,01042,6.98,1.0
1,R1,2,01042,3.99,2.0
2,R2,1,01107,3.49,1.4
3,R2,1,01107,3.49,1.4
4,R3,1,09999,5.00,0.5


,store,name
0,1042,Boston
1,1107,Hartford


**Idea 1 · `SELECT` computes new columns.** `ltrim(store, '0')` removes leading zeros. *We use it to make ids match across systems.*

In [2]:
tq("SELECT store AS \"before · store\", ltrim(store, '0') AS \"after · store_id\" FROM sales")

,before · store,after · store_id
0,01042,1042
1,01042,1042
2,01107,1107
3,01107,1107
4,09999,9999


**Idea 2 · `JOIN` matches rows from two tables.** A plain `JOIN` only keeps rows that match, so receipt R3 (unknown
store) silently **disappears**. A `LEFT JOIN` keeps every row, and `COALESCE(x, 'Unknown')` fills the gap.
*We use this so no sale is ever lost.*

In [3]:
display(tq("SELECT s.receipt, s.amount, st.name FROM sales s JOIN stores st ON st.store = ltrim(s.store, '0')"))
display(tq("SELECT s.receipt, s.amount, COALESCE(st.name, 'Unknown store') AS name FROM sales s LEFT JOIN stores st ON st.store = ltrim(s.store, '0')"))

,receipt,amount,name
0,R1,6.98,Boston
1,R1,3.99,Boston
2,R2,3.49,Hartford
3,R2,3.49,Hartford


,receipt,amount,name
0,R1,6.98,Boston
1,R1,3.99,Boston
2,R2,3.49,Hartford
3,R2,3.49,Hartford
4,R3,5.00,Unknown store


**Idea 3 · `GROUP BY` adds things up per group.** Notice Hartford shows 6.98, **double** its real 3.49, because of the duplicate line.

In [4]:
tq("SELECT ltrim(store, '0') AS store_id, SUM(amount) AS sales FROM sales GROUP BY 1")

,store_id,sales
0,1042,10.97
1,1107,6.98
2,9999,5.00


**Idea 4 · `ROW_NUMBER()` numbers copies, so we can keep one.** `ROW_NUMBER() OVER (PARTITION BY receipt, line)` gives
each copy of the same receipt line the numbers 1, 2, 3... Keeping `rn = 1` removes duplicates. *We use it in 5 steps.*

In [5]:
tq("SELECT receipt, line, amount, ROW_NUMBER() OVER (PARTITION BY store, receipt, line) AS rn FROM sales")

,receipt,line,amount,rn
0,R1,1,6.98,1
1,R1,2,3.99,1
2,R2,1,3.49,1
3,R2,1,3.49,2
4,R3,1,5.00,1


**Idea 5 · `LEAD()` looks at the next row.** Given dated prices, the next row's start date tells us when the current
price ended. *We use it to turn "effective from" dates into date ranges (costs, store history).*

In [6]:
toy.executescript('''CREATE TABLE cost (item TEXT, effective_from TEXT, unit_cost REAL);
INSERT INTO cost VALUES ('A', '2025-01-01', 1.00), ('A', '2025-07-01', 1.20), ('A', '2026-03-01', 1.25);''')
tq('''SELECT item, effective_from AS valid_from,
             COALESCE(date(LEAD(effective_from) OVER (PARTITION BY item ORDER BY effective_from), '-1 day'), '9999-12-31') AS valid_to,
             unit_cost
      FROM cost''')

,item,valid_from,valid_to,unit_cost
0,A,2025-01-01,2025-06-30,1.00
1,A,2025-07-01,2026-02-28,1.20
2,A,2026-03-01,9999-12-31,1.25


**Idea 6 · `WITH` names intermediate steps (a "CTE").** A long query becomes a readable list of steps. Every pipeline
file is written this way.

In [7]:
tq('''WITH numbered AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY store, receipt, line) AS rn FROM sales),   -- step 1
           deduped  AS (SELECT * FROM numbered WHERE rn = 1)                                                -- step 2
      SELECT ltrim(store, '0') AS store_id, SUM(amount) AS sales FROM deduped GROUP BY 1                   -- step 3''')

,store_id,sales
0,1042,10.97
1,1107,3.49
2,9999,5.00


Hartford is now 3.49. That's the whole toolkit: every transformation in this notebook is a combination of these six ideas.

---
# 3 · Setup: start with an empty warehouse

The demo uses **SQLite**, a small database built into Python. Each layer is a separate database file in `warehouse/`
(`bronze.db`, `silver.db`, `gold.db`), so SQL can say `bronze.pos_tlog_raw` or `gold.fct_sales_line`, just like
Databricks says `freshcart.bronze.pos_tlog_raw`.

The next cell **deletes any existing warehouse** and defines a few helpers used throughout:
`q(sql)` shows a query result as a table, `n(sql)` returns one number, `build(file, table)` runs one pipeline file,
`check(text, actual, expected)` prints ✅ or ❌.

In [8]:
import os, sys, json, csv
from pathlib import Path
from datetime import date

ROOT = Path.cwd()
while not (ROOT / "freshcart").exists() and ROOT != ROOT.parent:     # works from notebooks/ or the repo root
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from freshcart import config as C, bronze, contracts, quality, metrics
from freshcart.db import connect, render_sql, run_sql_file

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

con = connect(fresh_layers=C.LAYERS)          # delete warehouse/*.db and start from nothing

def q(sql):
    '''Run a query and show the result as a table.'''
    return pd.read_sql_query(render_sql(sql), con).convert_dtypes()

def n(sql):
    '''Run a query that returns a single value.'''
    return con.execute(render_sql(sql)).fetchone()[0]

def build(path, *tables):
    '''Run one pipeline SQL file, then report the row count of each table it built.'''
    run_sql_file(con, ROOT / path)
    for t in tables:
        print(f"▶️  built {t:<38} {n(f'SELECT COUNT(*) FROM {t}'):>8,} rows")

def check(text, actual, expected):
    print(f"{'✅' if actual == expected else '❌'}  {text}: {actual} (expected {expected})")

def layer_tables(layer):
    names = [r[0] for r in con.execute(f"SELECT name FROM {layer}.sqlite_master WHERE type = 'table' ORDER BY name")]
    return pd.DataFrame([(t, n(f'SELECT COUNT(*) FROM {layer}."{t}"')) for t in names], columns=[f"{layer} table", "rows"])

print("Tables in each layer right now:", {layer: len(layer_tables(layer)) for layer in C.LAYERS})
print("The pipeline's 'today' (as-of date):", C.AS_OF_DATE, "- pinned so every number is reproducible")

Tables in each layer right now: {'bronze': 0, 'silver': 0, 'gold': 0}
The pipeline's 'today' (as-of date): 2026-09-27 - pinned so every number is reproducible


Every layer is empty. **About "today":** in production the pipeline would use the real date. Here "today" is pinned to
**Sunday 27 September 2026**, so "last week" is always Sunday 20 – Saturday 26 September 2026 and every number in this
notebook is the same on every computer.

---
# 4 · BRONZE: copy every file in, change nothing

> 🎯 **Goal:** make every raw file queryable as a table, **exactly as delivered**, and remember where each row came from.

**The rules of bronze, and why:**

| Rule | Why |
|---|---|
| One table per feed (all 20 monthly till files → one table) | one place to look for each kind of data |
| **Every column is stored as text** | automatic type-guessing would quietly "fix" things: `01042` would become `1042`, `20250230` (30 Feb!) might become a date in March. Fixing is silver's job, done on purpose |
| Add 3 audit columns: `_source_file`, `_row_number`, `_ingested_at` | any number later can be traced back to the exact file and line |
| Load each file **once** (a log remembers loaded files) | re-running the pipeline never creates duplicates |
| Online orders (JSON) are kept whole | silver will take them apart |

> 💡 **Concept · Why not clean the data on the way in?** Because you will make mistakes in your cleaning rules. If bronze
> keeps the original, you fix the rule and rebuild. If bronze was already "cleaned", the original is lost for good.

### ▶️ Load all files into bronze

In [9]:
bronze.ingest(con);

  bronze.pos_tlog_raw              20 new of  20 files    98,192 rows
  bronze.ecom_order_raw            20 new of  20 files     8,288 rows
  bronze.product_master_raw         1 new of   1 files       162 rows
  bronze.merch_hierarchy_raw        1 new of   1 files        30 rows
  bronze.cost_history_raw           1 new of   1 files       114 rows
  bronze.store_master_raw           4 new of   4 files        35 rows
  bronze.ref_region_raw             1 new of   1 files         5 rows
  bronze.ref_store_format_raw       1 new of   1 files         4 rows
  bronze.ref_state_province_raw     1 new of   1 files         7 rows
  bronze.crm_member_raw             1 new of   1 files     1,367 rows
  bronze.promo_calendar_raw         1 new of   1 files       127 rows
  bronze.promo_item_raw             1 new of   1 files       575 rows
  bronze.fiscal_calendar_raw        1 new of   1 files     6,573 rows
  bronze.fx_rate_raw                1 new of   1 files       446 rows


  bronze.inventory_raw              4 new of   4 files    52,052 rows


### ✅ Check: bronze stores text, and loading twice adds nothing

In [10]:
display(q("SELECT QTY, typeof(QTY) AS stored_as, TRX_DT, typeof(TRX_DT) AS stored_as_, _source_file, _row_number FROM bronze.pos_tlog_raw LIMIT 2"))
before = n("SELECT COUNT(*) FROM bronze.pos_tlog_raw")
bronze.ingest(con, verbose=False)                                   # run the load a second time
check("Till rows after loading a second time", n("SELECT COUNT(*) FROM bronze.pos_tlog_raw"), before)
check("Files recorded in the ingestion log", n("SELECT COUNT(*) FROM bronze._ingestion_log"), 59)

,QTY,stored_as,TRX_DT,stored_as_,_source_file,_row_number
0,1.503,text,20250202,text,pos/pos_tlog_202502.csv,1
1,1.993,text,20250202,text,pos/pos_tlog_202502.csv,2


✅  Till rows after loading a second time: 98192 (expected 98192)
✅  Files recorded in the ingestion log: 59 (expected 59)


### The 15 bronze tables and what is wrong with each

Bronze changes nothing, so every problem is still in the data. This table is the **to-do list for silver**:

| Bronze table | From file(s) | One row = | Problems still in it | Fixed in |
|---|---|---|---|---|
| `pos_tlog_raw` | `pos/pos_tlog_*.csv` (20) | one line on a till receipt | zero-padded ids; text dates; S/R/V codes; returns positive; training till; a re-sent day; broken rows; card numbers; CAD | S7 |
| `ecom_order_raw` | `ecommerce/*.ndjson` (20) | one online order (whole JSON) | lines nested; timezone in timestamp; cancelled orders | S8 |
| `product_master_raw` | `erp/product_master.csv` | one product **per country** | every product twice; category codes; Canadian names in CAPITALS | S4 |
| `merch_hierarchy_raw` | `erp/merch_hierarchy.csv` | one category code | (a lookup table: it's the fix) | S4 |
| `cost_history_raw` | `erp/cost_history.csv` | one cost change | only a start date, no end date | S4 |
| `store_master_raw` | `stores/store_master_*.csv` (4) | one store per snapshot | codes; US dates; history hidden in 4 snapshots | S3 |
| `ref_store_format_raw`, `ref_region_raw`, `ref_state_province_raw` | `stores/ref_*.csv` | one code | (lookup tables: they're the fix) | S3 |
| `crm_member_raw` | `crm/crm_members_*.csv` | one loyalty card | **personal data**; codes; birthdays | S6 |
| `promo_calendar_raw`, `promo_item_raw` | `promotions/*.csv` | one promotion / one promo item | codes; text dates; empty strings | S5 |
| `fiscal_calendar_raw` | `finance/fiscal_calendar.csv` | one day | UK dates; labels like `FY2026` | S1 |
| `fx_rate_raw` | `finance/fx_rates.csv` | one weekday rate | no weekend rates | S2 |
| `inventory_raw` | `inventory/*.csv` (4) | one store-item-night | negative stock; text | S9 |

In [11]:
layer_tables("bronze")

,bronze table,rows
0,_ingestion_log,59
1,cost_history_raw,114
2,crm_member_raw,1367
3,ecom_order_raw,8288
4,fiscal_calendar_raw,6573
5,fx_rate_raw,446
6,inventory_raw,52052
7,merch_hierarchy_raw,30
8,pos_tlog_raw,98192
9,product_master_raw,162


---
# 5 · SILVER: clean and fix the data

Silver applies the same **kinds** of fix again and again. Learn these seven and you understand silver:

| Kind of fix | Example | Why |
|---|---|---|
| **Type** it | `'20260914'` → date `2026-09-14`; `'6.98'` → number 6.98 | you can't do date maths or add up text |
| **Standardise ids** | `01042`, `1042`, `9001` → no leading zeros | otherwise tables from different systems don't match |
| **Decode** codes | `NM` → `Neighborhood Market` | people (and AI) ask in words, not codes |
| **Remove duplicates** | the till day that was sent twice | otherwise sales are counted twice |
| **Flag, don't delete** | voided line → `line_type = 'Void'` | silver keeps every real event; gold decides what counts |
| **Quarantine** broken rows | no store number → set aside with the reason | never silently drop data, never let bad data through |
| **Protect personal data** | card number → irreversible code | privacy; the AI never sees it |

Each silver table is built by one SQL file in `pipeline/silver/`, rebuilt from bronze on every run.

---
## S1 · `silver.fiscal_calendar`

> 🎯 **Goal:** turn Finance's calendar file (text dates and labels) into a proper calendar table with real dates and numbers.

| | |
|---|---|
| 📥 **Input** | `bronze.fiscal_calendar_raw` |
| 📤 **Output** | `silver.fiscal_calendar` |
| 🧱 **One row =** | one calendar day (2010 – 2027) |
| ➡️ **Used later by** | S2 exchange rates, G1 `dim_date`, G2 `dim_store` |
| ❓ **Business questions it makes possible** | "last week", "period 3", "year to date vs last year" |

> 💡 **Concept · the retail (4-5-4) calendar.** Retailers don't report by calendar month, because months have different
> numbers of days and weekends: comparing September with last September might compare 4 Saturdays with 5. Instead the
> year is split into **weeks that always run Sunday to Saturday**, grouped into periods of 4, 5 and 4 weeks. The year
> starts on the Sunday nearest 1 February. So "day 50 of this year" is always the same weekday as "day 50 of last year".
> A year is 364 days, so every few years it gets a 53rd week to catch up.

### ⚠️ What's wrong with the input

In [12]:
q("SELECT CAL_DT, FISC_YR, FISC_QTR, FISC_PRD, FISC_PRD_NM, FISC_WK, FISC_WK_START FROM bronze.fiscal_calendar_raw WHERE CAL_DT IN ('31/01/2026', '01/02/2026', '08/02/2026')")

,CAL_DT,FISC_YR,FISC_QTR,FISC_PRD,FISC_PRD_NM,FISC_WK,FISC_WK_START
0,31/01/2026,FY2025,Q4,P12,Jan,W52,25/01/2026
1,01/02/2026,FY2026,Q1,P01,Feb,W01,01/02/2026
2,08/02/2026,FY2026,Q1,P01,Feb,W02,08/02/2026


* Dates are text, **day first** (`31/01/2026`). As text they don't even sort correctly (`01/02/2026` sorts before `31/01/2026`).
* Years, quarters, periods and weeks are **labels** (`FY2026`, `P01`, `W02`), so you can't say "the year before 2026".
* There's no easy way to line up "the same day last year".

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `calendar_date` | `CAL_DT` | parse as day/month/year → real date | `31/01/2026` → `2026-01-31` |
| `fiscal_year` | `FISC_YR` | drop the `FY`, make a number | `FY2026` → `2026` |
| `fiscal_quarter` | `FISC_QTR` | drop the `Q` | `Q1` → `1` |
| `fiscal_period` | `FISC_PRD` | drop the `P` | `P01` → `1` |
| `fiscal_period_name` | `FISC_PRD` + `FISC_PRD_NM` | join both labels | `P01`, `Feb` → `P01 Feb` |
| `fiscal_week` | `FISC_WK` | drop the `W` | `W02` → `2` |
| `fiscal_week_start_date` | `FISC_WK_START` | parse as day/month/year | `08/02/2026` → `2026-02-08` |
| `fiscal_day_of_year` | *new* | days since the first day of the fiscal year + 1 | `2026-02-08` → `8` |
| `day_of_week_name` | *new* | weekday of the date | `2026-02-08` → `Sunday` |

**Row rule:** if the same day arrives twice (a re-sent file), keep only the latest copy.

### 🔧 Transformation 1 · Parse the text dates
**What:** `try_to_date(CAL_DT, 'dd/MM/yyyy')` reads the text as day/month/year and returns a real date in the standard
`YYYY-MM-DD` format. **Why:** real dates sort, compare and support date maths ("the day before"). **Why state the format
explicitly?** The store master writes month first (`05/14/2016`). Guessing would swap day and month for the first 12
days of every month without any error.

In [13]:
q('''SELECT CAL_DT AS "before · CAL_DT", try_to_date(CAL_DT, 'dd/MM/yyyy') AS "after · calendar_date"
     FROM bronze.fiscal_calendar_raw WHERE CAL_DT IN ('31/01/2026', '01/02/2026', '08/02/2026')''')

,before · CAL_DT,after · calendar_date
0,31/01/2026,2026-01-31
1,01/02/2026,2026-02-01
2,08/02/2026,2026-02-08


### 🔧 Transformation 2 · Turn labels into numbers
**What:** `substr('FY2026', 3)` keeps the text from the 3rd character (`2026`), then `CAST(... AS INTEGER)` makes it a
number. Same for `Q`, `P`, `W`. **Why:** with numbers you can write `fiscal_year = 2026 - 1` to get last year.

In [14]:
q('''SELECT FISC_YR AS "before · FISC_YR", CAST(substr(FISC_YR, 3) AS INTEGER) AS "after · fiscal_year",
            FISC_PRD AS "before · FISC_PRD", CAST(substr(FISC_PRD, 2) AS INTEGER) AS "after · fiscal_period",
            FISC_WK AS "before · FISC_WK", CAST(substr(FISC_WK, 2) AS INTEGER) AS "after · fiscal_week"
     FROM bronze.fiscal_calendar_raw WHERE CAL_DT IN ('31/01/2026', '01/02/2026', '08/02/2026')''')

,before · FISC_YR,after · fiscal_year,before · FISC_PRD,after · fiscal_period,before · FISC_WK,after · fiscal_week
0,FY2025,2025,P12,12,W52,52
1,FY2026,2026,P01,1,W01,1
2,FY2026,2026,P01,1,W02,2


### 🔧 Transformation 3 · Add "fiscal day of year" and the weekday name
**What:** count the days from the first day of each fiscal year (`julianday()` turns a date into a day number, so
subtracting two gives the number of days between them). **Why:** "year to date vs last year" becomes simply "the same
`fiscal_day_of_year` range in both years", and because every fiscal year starts on a Sunday, the weekdays line up.

### ▶️ Build the table

In [15]:
build("pipeline/silver/01_fiscal_calendar.sql", "silver.fiscal_calendar")
q("SELECT * FROM silver.fiscal_calendar WHERE calendar_date IN ('2026-01-31', '2026-02-01', '2026-02-08')")

▶️  built silver.fiscal_calendar                    6,573 rows


,calendar_date,fiscal_year,fiscal_quarter,fiscal_period,fiscal_period_name,fiscal_week,fiscal_week_start_date,fiscal_day_of_year,day_of_week_name
0,2026-01-31,2025,4,12,P12 Jan,52,2026-01-25,364,Saturday
1,2026-02-01,2026,1,1,P01 Feb,1,2026-02-01,1,Sunday
2,2026-02-08,2026,1,1,P01 Feb,2,2026-02-08,8,Sunday


Notice 31 January 2026 is **day 364 of fiscal 2025**, and 1 February 2026 is day 1 of fiscal 2026.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/01_fiscal_calendar.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 01 · fiscal_calendar
-- Source : bronze.fiscal_calendar_raw   (Finance's file, text labels, DD/MM/YYYY dates)
-- Grain  : one row per calendar day
-- Steps  : 1. keep the latest delivery of each day (a re-sent file must not duplicate days)
--          2. parse DD/MM/YYYY strictly
--          3. turn text labels into numbers: 'FY2026' -> 2026, 'P01' -> 1, 'W34' -> 34
--          4. derive fiscal_day_of_year and the weekday name
-- =====================================================================================
DROP TABLE IF EXISTS silver.fiscal_calendar;
CREATE TABLE silver.fiscal_calendar (
  calendar_date          TEXT    PRIMARY KEY,
  fiscal_year            INTEGER NOT NULL,
  fiscal_quarter         INTEGER NOT NULL,
  fiscal_period          INTEGER NOT NULL,
  fiscal_period_name     TEXT    NOT NULL,
  fiscal_week            INTEGER NOT NULL,
  fiscal_week_start_date TEXT    NOT NULL,
  fiscal_day_of_year     INTEGER NOT NULL,
  day_of_week_name       TEXT    NOT NULL
);

INSERT INTO silver.fiscal_calendar
WITH latest AS (                                   -- step 1
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY CAL_DT ORDER BY _ingested_at DESC, _row_number DESC) AS rn
  FROM bronze.fiscal_calendar_raw
),
typed AS (                                         -- steps 2 and 3
  SELECT
    try_to_date(CAL_DT, 'dd/MM/yyyy')          AS calendar_date,
    CAST(substr(FISC_YR, 3) AS INTEGER)        AS fiscal_year,
    CAST(substr(FISC_QTR, 2) AS INTEGER)       AS fiscal_quarter,
    CAST(substr(FISC_PRD, 2) AS INTEGER)       AS fiscal_period,
    FISC_PRD || ' ' || FISC_PRD_NM             AS fiscal_period_name,
    CAST(substr(FISC_WK, 2) AS INTEGER)        AS fiscal_week,
    try_to_date(FISC_WK_START, 'dd/MM/yyyy')   AS fiscal_week_start_date
  FROM latest
  WHERE rn = 1
)
SELECT                                             -- step 4
  calendar_date,
  fiscal_year,
  fiscal_quarter,
  fiscal_period,
  fiscal_period_name,
  fiscal_week,
  fiscal_week_start_date,
  CAST(julianday(calendar_date)
       - julianday(MIN(calendar_date) OVER (PARTITION BY fiscal_year)) AS INTEGER) + 1 AS fiscal_day_of_year,
  CASE strftime('%w', calendar_date)
    WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday' WHEN '2' THEN 'Tuesday' WHEN '3' THEN 'Wednesday'
    WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday' WHEN '6' THEN 'Saturday' END AS day_of_week_name
FROM typed;
```
</details>

### ✅ Check

In [16]:
check("Days that appear more than once", n("SELECT COUNT(*) - COUNT(DISTINCT calendar_date) FROM silver.fiscal_calendar"), 0)
check("Dates that failed to parse", n("SELECT COUNT(*) FROM silver.fiscal_calendar WHERE calendar_date IS NULL"), 0)
check("Fiscal years that don't start on a Sunday", n("SELECT COUNT(*) FROM silver.fiscal_calendar WHERE fiscal_day_of_year = 1 AND day_of_week_name <> 'Sunday'"), 0)

✅  Days that appear more than once: 0 (expected 0)
✅  Dates that failed to parse: 0 (expected 0)
✅  Fiscal years that don't start on a Sunday: 0 (expected 0)


### 📌 Recap
* Text dates → real dates, with the day-first format stated explicitly.
* Labels (`FY2026`, `P01`) → numbers you can calculate with.
* Added `fiscal_day_of_year`, which makes "same period last year" easy and weekday-aligned.

---
## S2 · `silver.fx_rate_daily`

> 🎯 **Goal:** have a Canadian-dollar → US-dollar exchange rate for **every** calendar day, so every Canadian sale can be converted.

| | |
|---|---|
| 📥 **Input** | `bronze.fx_rate_raw` + `silver.fiscal_calendar` (S1) |
| 📤 **Output** | `silver.fx_rate_daily` |
| 🧱 **One row =** | one day × one currency (CAD or USD) |
| ➡️ **Used later by** | S7 till sales, S8 online sales |
| ❓ **Business questions it makes possible** | any sales total across the US and Canada (group reporting is in USD) |

### ⚠️ What's wrong with the input
Rates are only published on **business days**. There is nothing for Saturday 13 and Sunday 14 September 2025, but
the Toronto store is open on weekends:

In [17]:
q("SELECT RATE_DT, FROM_CCY, TO_CCY, RATE FROM bronze.fx_rate_raw WHERE RATE_DT BETWEEN '2025-09-11' AND '2025-09-16'")

,RATE_DT,FROM_CCY,TO_CCY,RATE
0,2025-09-11,CAD,USD,0.7472
1,2025-09-12,CAD,USD,0.7508
2,2025-09-15,CAD,USD,0.7447
3,2025-09-16,CAD,USD,0.7475


If we joined weekend sales to this table they would find **no rate** and either disappear or become NULL.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `rate_date` | every day of `silver.fiscal_calendar` | one row per calendar day, not just business days | (no row for Sat 13th) → `2025-09-13` |
| `from_currency` | `FROM_CCY` | upper case; plus an extra `USD` row per day | `CAD` → `CAD`; *new* `USD` |
| `rate_to_usd` | `RATE` | text → number; on days with no rate, use the **last published** rate; USD = 1.0 | (none on Sat) → `0.7508` (Friday's) |
| `is_carried_forward` | *new* | 1 if the rate was copied from an earlier day | Sat 13th → `1` |

### 🔧 Transformation 1 · One row for every day, carrying the last rate forward
**What:** start from the calendar (every day), and for each day take the most recent published rate **on or before**
that day. **Why:** a Saturday sale is converted at Friday's rate, the standard accounting practice.
**Why the flag?** So nobody mistakes a copied rate for a published one.

> 💡 **Concept · forward fill.** Filling a gap with the last known value is called *forward fill* (or "last observation
> carried forward"). It's the right choice for things that stay valid until they change, like prices and rates.

### 🔧 Transformation 2 · Add USD → USD = 1.0
**What:** add a row with rate 1.0 for US dollars on every day. **Why:** now **every** sale, US or Canadian, is converted
the same way (`amount × rate`). No special case for US stores means one less place for a bug.

### ▶️ Build the table

In [18]:
build("pipeline/silver/02_fx_rate_daily.sql", "silver.fx_rate_daily")
q("SELECT * FROM silver.fx_rate_daily WHERE rate_date BETWEEN '2025-09-11' AND '2025-09-16' ORDER BY from_currency DESC, rate_date")

▶️  built silver.fx_rate_daily                      1,268 rows


,rate_date,from_currency,rate_to_usd,is_carried_forward
0,2025-09-11,USD,1.0,0
1,2025-09-12,USD,1.0,0
2,2025-09-13,USD,1.0,0
3,2025-09-14,USD,1.0,0
4,2025-09-15,USD,1.0,0
5,2025-09-16,USD,1.0,0
6,2025-09-11,CAD,0.7472,0
7,2025-09-12,CAD,0.7508,0
8,2025-09-13,CAD,0.7508,1
9,2025-09-14,CAD,0.7508,1


Saturday 13 and Sunday 14 now have Friday's rate (0.7508), flagged with `is_carried_forward = 1`.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/02_fx_rate_daily.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 02 · fx_rate_daily
-- Source : bronze.fx_rate_raw   (CAD->USD, published on business days only)
-- Grain  : one row per calendar day per currency
-- Steps  : 1. type the published rates
--          2. build a row for EVERY calendar day, carrying the last published rate forward
--             over weekends and holidays (a Saturday sale still needs a rate)
--          3. add USD->USD = 1.0 so every sale can use the same join
-- =====================================================================================
DROP TABLE IF EXISTS silver.fx_rate_daily;
CREATE TABLE silver.fx_rate_daily (
  rate_date          TEXT    NOT NULL,
  from_currency      TEXT    NOT NULL,
  rate_to_usd        REAL    NOT NULL,
  is_carried_forward INTEGER NOT NULL,      -- 1 when no rate was published that day
  PRIMARY KEY (rate_date, from_currency)
);

DROP TABLE IF EXISTS temp._published;
CREATE TEMP TABLE _published AS                                        -- step 1
SELECT try_to_date(RATE_DT, 'yyyy-MM-dd') AS rate_date,
       upper(FROM_CCY)                    AS from_currency,
       CAST(RATE AS REAL)                 AS rate
FROM bronze.fx_rate_raw
WHERE upper(TO_CCY) = 'USD';

INSERT INTO silver.fx_rate_daily
SELECT                                                                 -- step 2
  c.calendar_date,
  'CAD',
  (SELECT p.rate FROM _published p
   WHERE p.from_currency = 'CAD' AND p.rate_date <= c.calendar_date
   ORDER BY p.rate_date DESC LIMIT 1),
  CASE WHEN EXISTS (SELECT 1 FROM _published p
                    WHERE p.from_currency = 'CAD' AND p.rate_date = c.calendar_date) THEN 0 ELSE 1 END
FROM silver.fiscal_calendar c
WHERE c.calendar_date BETWEEN (SELECT MIN(rate_date) FROM _published) AND '${AS_OF_DATE}';

INSERT INTO silver.fx_rate_daily                                       -- step 3
SELECT calendar_date, 'USD', 1.0, 0
FROM silver.fiscal_calendar
WHERE calendar_date BETWEEN (SELECT MIN(rate_date) FROM _published) AND '${AS_OF_DATE}';
```
</details>

### ✅ Check

In [19]:
check("Days up to 'today' with no CAD rate", n('''SELECT COUNT(*) FROM silver.fiscal_calendar c
      WHERE c.calendar_date BETWEEN (SELECT MIN(rate_date) FROM silver.fx_rate_daily) AND '${AS_OF_DATE}'
        AND NOT EXISTS (SELECT 1 FROM silver.fx_rate_daily f WHERE f.rate_date = c.calendar_date AND f.from_currency = 'CAD')'''), 0)
print("Days where the CAD rate was carried forward:", n("SELECT SUM(is_carried_forward) FROM silver.fx_rate_daily WHERE from_currency = 'CAD'"))

✅  Days up to 'today' with no CAD rate: 0 (expected 0)
Days where the CAD rate was carried forward: 188


### 📌 Recap
* Weekday-only rates → a rate for every day (forward fill), flagged when copied.
* A USD = 1.0 row so all sales use one conversion rule.

---
## S3 · `silver.store_history`

> 🎯 **Goal:** one clean, decoded record per store, and a **new version** whenever a store's details change (so history isn't lost).

| | |
|---|---|
| 📥 **Input** | `bronze.store_master_raw` (4 snapshots) + `ref_store_format_raw`, `ref_region_raw`, `ref_state_province_raw` |
| 📤 **Output** | `silver.store_history` |
| 🧱 **One row =** | one store × one **version** of its details |
| ➡️ **Used later by** | G2 `dim_store` |
| ❓ **Business questions it makes possible** | "sales by region / format", "which stores opened in FY2025?", "what format was Boston when it made this sale?" |

### ⚠️ What's wrong with the input
The store master is sent as a full **snapshot** every time something changes (4 files). Here is Boston Seaport and
Atlanta Buckhead in all four:

In [20]:
q('''SELECT _source_file, STR_NBR, STR_NM, FMT_CD, RGN_CD, ST_CD, OPEN_DT, CLOSE_DT, SQFT
     FROM bronze.store_master_raw WHERE STR_NBR IN ('01042', '02211') ORDER BY STR_NBR, _source_file''')

,_source_file,STR_NBR,STR_NM,FMT_CD,RGN_CD,ST_CD,OPEN_DT,CLOSE_DT,SQFT
0,stores/store_master_20250201.csv,01042,FreshCart Boston Seaport,NM,02,MA,05/14/2016,,38000
1,stores/store_master_20250601.csv,01042,FreshCart Boston Seaport,NM,02,MA,05/14/2016,,38000
2,stores/store_master_20260201.csv,01042,FreshCart Boston Seaport,SC,02,MA,05/14/2016,,52000
3,stores/store_master_20260601.csv,01042,FreshCart Boston Seaport,SC,02,MA,05/14/2016,,52000
4,stores/store_master_20250201.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,03/18/2017,,8800
5,stores/store_master_20250601.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,03/18/2017,,8800
6,stores/store_master_20260201.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,03/18/2017,,8800
7,stores/store_master_20260601.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,03/18/2017,05/30/2026,8800


* Format, region and state are **codes** (`NM`, `02`, `MA`). The meanings are in three separate lookup files.
* Dates are **US style, month first** (`05/14/2016`), the opposite of Finance's calendar.
* Store numbers have **leading zeros** (`01042`), but the website and ERP say `1042`.
* The snapshot date is only in the **file name**.
* The same store appears 4 times, even though Boston only **changed once** (Neighborhood Market → Supercenter, 38,000 →
  52,000 sq ft, in the 2026-02-01 snapshot) and Buckhead only once (a close date appeared).

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `store_id` | `STR_NBR` | remove leading zeros | `01042` → `1042` |
| `store_name` | `STR_NM` | unchanged | `FreshCart Boston Seaport` |
| `store_format` | `FMT_CD` | look up in `ref_store_format_raw` | `NM` → `Neighborhood Market` |
| `city` | `CITY` | unchanged | `Boston` |
| `state_province` | `ST_CD` | look up in `ref_state_province_raw` | `MA` → `Massachusetts` |
| `country` | `CNTRY` | decode | `US` → `United States` |
| `region` | `RGN_CD` | look up in `ref_region_raw` | `02` → `Northeast` |
| `open_date` | `OPEN_DT` | parse as **month**/day/year | `05/14/2016` → `2016-05-14` |
| `close_date` | `CLOSE_DT` | parse; empty → NULL (still open) | `05/30/2026` → `2026-05-30` |
| `selling_area_sqft` | `SQFT` | text → whole number | `52000` → `52000` |
| `valid_from` | *new* (file name) | date this version started | `store_master_20260201.csv` → `2026-02-01` |
| `valid_to` | *new* | day before the next version starts; `9999-12-31` if none | → `2026-01-31` |
| `is_current` | *new* | 1 for the latest version | → `1` |

**Row rule:** a snapshot becomes a new version **only if something changed**; identical snapshots are dropped.

### 🔧 Transformation 1 · Decode, standardise and parse (all in one pass)
**What:** `LEFT JOIN` each lookup table to swap codes for names; `ltrim(STR_NBR, '0')` for the id; `try_to_date(OPEN_DT,
'MM/dd/yyyy')` for dates; read the snapshot date from characters 8 before `.csv` in the file name.
**Why:** people ask "Northeast", not "02"; the id must match sales; dates must be real dates. **Why `LEFT JOIN`?** If a
code were missing from a lookup, a plain join would make the whole store vanish.

In [21]:
q('''SELECT substr(s._source_file, -12, 8) AS "before · file date", try_to_date(substr(s._source_file, -12, 8), 'yyyyMMdd') AS "after · snapshot_date",
            s.STR_NBR AS "before · STR_NBR", ltrim(s.STR_NBR, '0') AS "after · store_id",
            s.FMT_CD AS "before · FMT_CD", f.FMT_NM AS "after · store_format",
            s.RGN_CD AS "before · RGN_CD", r.RGN_NM AS "after · region",
            s.OPEN_DT AS "before · OPEN_DT", try_to_date(s.OPEN_DT, 'MM/dd/yyyy') AS "after · open_date"
     FROM bronze.store_master_raw s
     LEFT JOIN bronze.ref_store_format_raw f ON f.FMT_CD = s.FMT_CD
     LEFT JOIN bronze.ref_region_raw r ON r.RGN_CD = s.RGN_CD
     WHERE s.STR_NBR = '01042' ORDER BY 1''')

,before · file date,after · snapshot_date,before · STR_NBR,after · store_id,before · FMT_CD,after · store_format,before · RGN_CD,after · region,before · OPEN_DT,after · open_date
0,20250201,2025-02-01,01042,1042,NM,Neighborhood Market,02,Northeast,05/14/2016,2016-05-14
1,20250601,2025-06-01,01042,1042,NM,Neighborhood Market,02,Northeast,05/14/2016,2016-05-14
2,20260201,2026-02-01,01042,1042,SC,Supercenter,02,Northeast,05/14/2016,2016-05-14
3,20260601,2026-06-01,01042,1042,SC,Supercenter,02,Northeast,05/14/2016,2016-05-14


> 💡 **Concept · Slowly Changing Dimension, type 2 (SCD2).** Store details change *slowly* (a remodel, a closure). You
> can either **overwrite** the old details (type 1: simple, but then Boston's 2025 sales would look like Supercenter
> sales), or keep **one row per version**, each with the dates it was valid (type 2). We keep versions, so any past sale
> can be matched to the store's details **on the day of the sale**.

### 🔧 Transformation 2 · Keep a snapshot only when something changed
**What:** glue all the store's details into one string (`attr_hash`), then use `LAG()` (the *previous* row) to compare
each snapshot with the one before. Unchanged snapshots are dropped.
**Why:** otherwise Boston would have 4 "versions" that are really only 2.

In [22]:
q('''WITH snap AS (
       SELECT ltrim(STR_NBR, '0') AS store_id, substr(_source_file, -12, 8) AS snapshot, FMT_CD, SQFT, CLOSE_DT,
              STR_NM || '|' || FMT_CD || '|' || RGN_CD || '|' || CLOSE_DT || '|' || SQFT AS attr_hash
       FROM bronze.store_master_raw WHERE STR_NBR = '01042')
     SELECT store_id, snapshot, FMT_CD, SQFT,
            LAG(attr_hash) OVER (ORDER BY snapshot) = attr_hash AS same_as_previous,
            CASE WHEN LAG(attr_hash) OVER (ORDER BY snapshot) = attr_hash THEN 'drop' ELSE 'keep as a version' END AS decision
     FROM snap ORDER BY snapshot''')

,store_id,snapshot,FMT_CD,SQFT,same_as_previous,decision
0,1042,20250201,NM,38000,<NA>,keep as a version
1,1042,20250601,NM,38000,1,drop
2,1042,20260201,SC,52000,0,keep as a version
3,1042,20260601,SC,52000,1,drop


### 🔧 Transformation 3 · Give each version a date range
**What:** `valid_from` = when the version started; `valid_to` = the day before the **next** version starts (`LEAD()`, the
*next* row), or `9999-12-31` ("still valid") for the latest, which is flagged `is_current = 1`.
**Why:** with a range, "which version was valid on 15 Dec 2025?" is a simple `BETWEEN valid_from AND valid_to`.

### ▶️ Build the table

In [23]:
build("pipeline/silver/03_store_history.sql", "silver.store_history")
q('''SELECT store_id, store_name, store_format, region, selling_area_sqft, close_date, valid_from, valid_to, is_current
     FROM silver.store_history WHERE store_id IN ('1042', '2211') ORDER BY store_id, valid_from''')

▶️  built silver.store_history                         11 rows


,store_id,store_name,store_format,region,selling_area_sqft,close_date,valid_from,valid_to,is_current
0,1042,FreshCart Boston Seaport,Neighborhood Market,Northeast,38000,<NA>,2016-05-14,2026-01-31,0
1,1042,FreshCart Boston Seaport,Supercenter,Northeast,52000,<NA>,2026-02-01,9999-12-31,1
2,2211,FreshCart Atlanta Buckhead,Express,Southeast,8800,<NA>,2017-03-18,2026-05-31,0
3,2211,FreshCart Atlanta Buckhead,Express,Southeast,8800,2026-05-30,2026-06-01,9999-12-31,1


4 snapshots × 9 stores (35 rows) became 11 versions: 2 for Boston (remodel), 2 for Buckhead (closure), 1 for everyone else.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/03_store_history.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 03 · store_history   (slowly changing dimension, type 2)
-- Source : bronze.store_master_raw (4 monthly snapshots) + three code lookups
-- Grain  : one row per store per VERSION of its attributes
-- Steps  : 1. decode codes: FMT_CD 'NM' -> 'Neighborhood Market', RGN_CD '02' -> 'Northeast',
--             ST_CD 'MA' -> 'Massachusetts', CNTRY 'US' -> 'United States'
--          2. normalise ids ('01042' -> '1042') and parse US-style MM/DD/YYYY dates
--          3. read the snapshot date from the file name (store_master_20260201.csv)
--          4. keep a snapshot only when something changed since the previous snapshot
--          5. give each version a valid_from / valid_to range and flag the current one
-- Why    : history answers "what format was Boston when it sold this?"; gold uses
--          the current version, silver keeps them all.
-- =====================================================================================
DROP TABLE IF EXISTS silver.store_history;
CREATE TABLE silver.store_history (
  store_id          TEXT    NOT NULL,
  store_name        TEXT    NOT NULL,
  store_format      TEXT    NOT NULL,
  city              TEXT,
  state_province    TEXT,
  country           TEXT,
  region            TEXT    NOT NULL,
  open_date         TEXT,
  close_date        TEXT,
  selling_area_sqft INTEGER,
  valid_from        TEXT    NOT NULL,
  valid_to          TEXT    NOT NULL,
  is_current        INTEGER NOT NULL,
  PRIMARY KEY (store_id, valid_from)
);

INSERT INTO silver.store_history
WITH snap AS (
  SELECT
    ltrim(s.STR_NBR, '0')                                    AS store_id,          -- step 2
    s.STR_NM                                                 AS store_name,
    f.FMT_NM                                                 AS store_format,      -- step 1
    s.CITY                                                   AS city,
    st.ST_NM                                                 AS state_province,
    CASE s.CNTRY WHEN 'US' THEN 'United States' WHEN 'CA' THEN 'Canada' END AS country,
    r.RGN_NM                                                 AS region,
    try_to_date(s.OPEN_DT, 'MM/dd/yyyy')                     AS open_date,         -- step 2
    try_to_date(NULLIF(s.CLOSE_DT, ''), 'MM/dd/yyyy')        AS close_date,
    CAST(s.SQFT AS INTEGER)                                  AS selling_area_sqft,
    try_to_date(substr(s._source_file, -12, 8), 'yyyyMMdd')  AS snapshot_date      -- step 3
  FROM bronze.store_master_raw s
  LEFT JOIN bronze.ref_store_format_raw   f  ON f.FMT_CD  = s.FMT_CD
  LEFT JOIN bronze.ref_region_raw         r  ON r.RGN_CD  = s.RGN_CD
  LEFT JOIN bronze.ref_state_province_raw st ON st.ST_CD  = s.ST_CD
),
hashed AS (
  SELECT *,
         store_name || '|' || store_format || '|' || COALESCE(city, '') || '|' || COALESCE(state_province, '') || '|' ||
         region || '|' || COALESCE(open_date, '') || '|' || COALESCE(close_date, '') || '|' ||
         COALESCE(selling_area_sqft, '') AS attr_hash
  FROM snap
),
changes AS (                                                                         -- step 4
  SELECT *, LAG(attr_hash) OVER (PARTITION BY store_id ORDER BY snapshot_date) AS prev_hash
  FROM hashed
),
versions AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY snapshot_date) AS version_no
  FROM changes
  WHERE prev_hash IS NULL OR prev_hash <> attr_hash
),
ranged AS (                                                                          -- step 5
  SELECT *,
         CASE WHEN version_no = 1 THEN MIN(open_date, snapshot_date) ELSE snapshot_date END AS valid_from
  FROM versions
)
SELECT
  store_id, store_name, store_format, city, state_province, country, region,
  open_date, close_date, selling_area_sqft,
  valid_from,
  COALESCE(date(LEAD(valid_from) OVER (PARTITION BY store_id ORDER BY valid_from), '-1 day'), '9999-12-31') AS valid_to,
  CASE WHEN LEAD(valid_from) OVER (PARTITION BY store_id ORDER BY valid_from) IS NULL THEN 1 ELSE 0 END  AS is_current
FROM ranged;
```
</details>

### ✅ Check

In [24]:
check("Stores without exactly one current version", n("SELECT COUNT(*) FROM (SELECT store_id FROM silver.store_history GROUP BY store_id HAVING SUM(is_current) <> 1)"), 0)
check("Codes that failed to decode", n("SELECT COUNT(*) FROM silver.store_history WHERE store_format IS NULL OR region IS NULL OR state_province IS NULL"), 0)
display(q('''SELECT '2025-12-15' AS sale_date, store_format AS boston_format_that_day FROM silver.store_history WHERE store_id = '1042' AND '2025-12-15' BETWEEN valid_from AND valid_to
             UNION ALL SELECT '2026-09-14', store_format FROM silver.store_history WHERE store_id = '1042' AND '2026-09-14' BETWEEN valid_from AND valid_to'''))

✅  Stores without exactly one current version: 0 (expected 0)
✅  Codes that failed to decode: 0 (expected 0)


,sale_date,boston_format_that_day
0,2025-12-15,Neighborhood Market
1,2026-09-14,Supercenter


### 📌 Recap
* Codes → names (via lookups), ids without zeros, US dates parsed as month-first.
* 4 snapshots → only the versions where something **changed** (SCD2), each with a validity range.
* Now "what was the store like on the day of the sale?" has exactly one answer.

---
## S4 · `silver.product  and  silver.product_cost`

> 🎯 **Goal:** exactly **one** row per product with its category in words, plus each product's cost for any given date.

| | |
|---|---|
| 📥 **Input** | `bronze.product_master_raw`, `bronze.merch_hierarchy_raw`, `bronze.cost_history_raw` |
| 📤 **Output** | `silver.product  and  silver.product_cost` |
| 🧱 **One row =** | `product`: one product · `product_cost`: one product × one cost period |
| ➡️ **Used later by** | G3 `dim_product`, G6 sales (cost → margin), G7 stock (value) |
| ❓ **Business questions it makes possible** | "sales by category", "margin by subcategory", "out of stock in fresh" |

### ⚠️ What's wrong with the input
The ERP stores each product **once per country** (sales org `US01` and `CA01`):

In [25]:
display(q("SELECT MATNR, VKORG, MAKTX, MATKL, MEINS, STATUS FROM bronze.product_master_raw WHERE MATNR IN ('4471023', '4471024', '4492050') ORDER BY MATNR, VKORG"))
print("rows in the raw product master:", n("SELECT COUNT(*) FROM bronze.product_master_raw"),
      "| actual products:", n("SELECT COUNT(DISTINCT MATNR) FROM bronze.product_master_raw"))

,MATNR,VKORG,MAKTX,MATKL,MEINS,STATUS
0,4471023,CA01,CRUNCHY SEA SALT CHIPS 200G,C0412,EA,A
1,4471023,US01,Crunchy Sea Salt Chips 200g,C0412,EA,A
2,4471024,CA01,BBQ KETTLE CHIPS 200G,C0412,EA,A
3,4471024,US01,BBQ Kettle Chips 200g,C0412,EA,D
4,4492050,CA01,MAPLE BAKED BEANS 398ML,C0432,EA,A


rows in the raw product master: 162 | actual products: 82


* **Most products appear twice.** This is the most dangerous problem in the whole dataset (see below).
* The category is a code (`C0412`); unit and status are codes (`EA`, `KG`, `A`, `D`).
* The Canadian copy is in CAPITALS and never records discontinued items (`4471024` is `D` in the US, `A` in Canada).
* Two products exist **only** in Canada (`4492050`).
* Costs only have a start date (`COST_EFF_DT`); the end is implied by the next change.

> 💡 **Concept · grain and "fan-out".** The **grain** of a table is what one row means. We need "one row per
> product", but the raw table is "one row per product per country". Join sales to it and every sale matches **two**
> product rows, so every sale is counted **twice**. This is called *fan-out*, and it produces no error, just wrong numbers:

In [26]:
q('''SELECT 'Till sales of chips 4471023, counted directly' AS how, COUNT(*) AS lines, ROUND(SUM(CAST(EXT_AMT AS REAL)), 2) AS sales
     FROM bronze.pos_tlog_raw WHERE ltrim(ITM_ID, '0') = '4471023'
     UNION ALL
     SELECT 'Same sales, after joining the raw product master', COUNT(*), ROUND(SUM(CAST(s.EXT_AMT AS REAL)), 2)
     FROM bronze.pos_tlog_raw s JOIN bronze.product_master_raw p ON ltrim(p.MATNR, '0') = ltrim(s.ITM_ID, '0')
     WHERE ltrim(s.ITM_ID, '0') = '4471023' ''')

,how,lines,sales
0,"Till sales of chips 4471023, counted directly",1314,6473.19
1,"Same sales, after joining the raw product master",2628,12946.38


Sales doubled. Fixing the grain to one row per product prevents this.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `product_id` | `MATNR` | remove leading zeros | `000004471023` (tills) / `4471023` → `4471023` |
| `product_name` | `MAKTX` | US copy as-is; Canadian-only copy turned into Title Case | `MAPLE BAKED BEANS 398ML` → `Maple Baked Beans 398ml` |
| `brand` | `BRAND_NM` | unchanged | `Crunchwell` |
| `department, category, subcategory` | `MATKL` | look up in `merch_hierarchy_raw` | `C0412` → `Grocery` / `Snacks` / `Chips and Crisps` |
| `unit_of_measure` | `MEINS` | decode | `EA` → `Each`, `KG` → `Kilogram` |
| `product_status` | `STATUS` | decode | `A` → `Active`, `D` → `Discontinued` |
| `material_group_code` | `MATKL` | kept for traceability | `C0412` |
| `source_sales_org` | `VKORG` | which copy was chosen | `US01` |

**Row rule:** one row per product: take the `US01` copy; use `CA01` only when there is no US copy.

### 🔧 Transformation 1 · Remove the duplicate copies, preferring the US record
**What:** `ROW_NUMBER() OVER (PARTITION BY product ORDER BY (US01 first))` numbers the copies of each product with the US
copy as 1; we keep row 1. **Why US01?** It's the master record, the one that correctly marks discontinued products.
**Why not just pick any?** Picking the Canadian copy of `4471024` would make a discontinued product look active, and it
would then show up as "out of stock".

In [27]:
q('''SELECT ltrim(MATNR, '0') AS product_id, VKORG, MAKTX, STATUS,
            ROW_NUMBER() OVER (PARTITION BY ltrim(MATNR, '0') ORDER BY CASE VKORG WHEN 'US01' THEN 1 ELSE 2 END) AS rn,
            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ltrim(MATNR, '0') ORDER BY CASE VKORG WHEN 'US01' THEN 1 ELSE 2 END) = 1
                 THEN 'keep' ELSE 'drop' END AS decision
     FROM bronze.product_master_raw WHERE MATNR IN ('4471023', '4471024', '4492050') ORDER BY 1, rn''')

,product_id,VKORG,MAKTX,STATUS,rn,decision
0,4471023,US01,Crunchy Sea Salt Chips 200g,A,1,keep
1,4471023,CA01,CRUNCHY SEA SALT CHIPS 200G,A,2,drop
2,4471024,US01,BBQ Kettle Chips 200g,D,1,keep
3,4471024,CA01,BBQ KETTLE CHIPS 200G,A,2,drop
4,4492050,CA01,MAPLE BAKED BEANS 398ML,A,1,keep


### 🔧 Transformation 2 · Decode category, unit and status
**What:** join `MATKL` to the hierarchy lookup; `CASE` for unit and status. **Why:** "margin by Snacks subcategory" needs
the words `Snacks` and `Chips and Crisps` to exist in the data.

In [28]:
q('''SELECT p.MATKL AS "before · MATKL", h.DEPT_NM AS "after · department", h.CAT_NM AS "after · category", h.SUBCAT_NM AS "after · subcategory",
            p.MEINS AS "before · MEINS", CASE p.MEINS WHEN 'EA' THEN 'Each' WHEN 'KG' THEN 'Kilogram' END AS "after · unit_of_measure"
     FROM bronze.product_master_raw p LEFT JOIN bronze.merch_hierarchy_raw h ON h.MATKL = p.MATKL
     WHERE p.VKORG = 'US01' AND p.MATNR IN ('4471023', '93321', '5012')''')

,before · MATKL,after · department,after · category,after · subcategory,before · MEINS,after · unit_of_measure
0,P0101,Produce,Fresh Fruit,Bananas and Tropical,KG,Kilogram
1,E0101,Dairy and Eggs,Milk and Cream,Milk,EA,Each
2,C0412,Grocery,Snacks,Chips and Crisps,EA,Each


### 🔧 Transformation 3 · Costs: from start dates to date ranges (`product_cost`)
**What:** for each cost, `valid_to` = the day before the next cost starts (`LEAD()`), or `9999-12-31`.
**Why:** costs change during the year. A sale in March must use March's cost, or the margin is wrong.

In [29]:
q('''SELECT MATNR, COST_EFF_DT AS "before · COST_EFF_DT", STD_COST,
            try_to_date(COST_EFF_DT, 'yyyyMMdd') AS "after · valid_from",
            COALESCE(date(LEAD(try_to_date(COST_EFF_DT, 'yyyyMMdd')) OVER (PARTITION BY MATNR ORDER BY COST_EFF_DT), '-1 day'), '9999-12-31') AS "after · valid_to"
     FROM bronze.cost_history_raw WHERE MATNR IN ('20101', '20201')''')

,MATNR,before · COST_EFF_DT,STD_COST,after · valid_from,after · valid_to
0,20101,20240101,1.10,2024-01-01,2025-12-27
1,20101,20251228,1.14,2025-12-28,9999-12-31
2,20201,20240101,2.42,2024-01-01,2026-07-12
3,20201,20260713,2.53,2026-07-13,9999-12-31


### ▶️ Build the tables

In [30]:
build("pipeline/silver/04_product.sql", "silver.product", "silver.product_cost")
q("SELECT product_id, product_name, brand, department, category, subcategory, unit_of_measure, product_status, source_sales_org FROM silver.product WHERE product_id IN ('4471023', '4471024', '4492050')")

▶️  built silver.product                               82 rows
▶️  built silver.product_cost                         114 rows


,product_id,product_name,brand,department,category,subcategory,unit_of_measure,product_status,source_sales_org
0,4471023,Crunchy Sea Salt Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,Active,US01
1,4471024,BBQ Kettle Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,Discontinued,US01
2,4492050,Maple Baked Beans 398ml,Pantry Lane,Grocery,Pantry,Canned Goods,Each,Active,CA01


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/04_product.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 04 · product  and  product_cost
-- Source : bronze.product_master_raw  (one row per SKU PER SALES ORG: US01 and CA01)
--          bronze.merch_hierarchy_raw (material group code -> names)
--          bronze.cost_history_raw    (standard cost with effective dates)
-- Grain  : product      -> one row per SKU
--          product_cost -> one row per SKU per cost version
-- Steps  : 1. normalise the SKU (no leading zeros) so POS, online and ERP ids match
--          2. DEDUPLICATE: prefer the US01 record, fall back to CA01 for Canada-only items.
--             Without this, every sale joins to two product rows and sales double.
--          3. decode MATKL 'C0412' -> Grocery / Snacks / Chips and Crisps
--          4. decode unit and status codes; fix the upper-case CA01 descriptions
--          5. turn cost effective dates into valid_from / valid_to ranges
-- =====================================================================================
DROP TABLE IF EXISTS silver.product;
CREATE TABLE silver.product (
  product_id          TEXT PRIMARY KEY,
  product_name        TEXT NOT NULL,
  brand               TEXT,
  department          TEXT NOT NULL,
  category            TEXT NOT NULL,
  subcategory         TEXT NOT NULL,
  unit_of_measure     TEXT NOT NULL,
  product_status      TEXT NOT NULL,
  material_group_code TEXT NOT NULL,
  source_sales_org    TEXT NOT NULL
);

INSERT INTO silver.product
WITH ranked AS (
  SELECT p.*,
         ltrim(p.MATNR, '0') AS product_id,                                             -- step 1
         ROW_NUMBER() OVER (PARTITION BY ltrim(p.MATNR, '0')
                            ORDER BY CASE p.VKORG WHEN 'US01' THEN 1 ELSE 2 END,
                                     p._ingested_at DESC) AS rn                          -- step 2
  FROM bronze.product_master_raw p
)
SELECT
  r.product_id,
  CASE WHEN r.VKORG = 'US01' THEN r.MAKTX ELSE initcap(r.MAKTX) END AS product_name,    -- step 4
  r.BRAND_NM,
  h.DEPT_NM, h.CAT_NM, h.SUBCAT_NM,                                                      -- step 3
  CASE r.MEINS WHEN 'EA' THEN 'Each' WHEN 'KG' THEN 'Kilogram' ELSE r.MEINS END,
  CASE r.STATUS WHEN 'A' THEN 'Active' WHEN 'D' THEN 'Discontinued' ELSE 'Unknown' END,
  r.MATKL,
  r.VKORG
FROM ranked r
LEFT JOIN bronze.merch_hierarchy_raw h ON h.MATKL = r.MATKL
WHERE r.rn = 1;

DROP TABLE IF EXISTS silver.product_cost;
CREATE TABLE silver.product_cost (
  product_id    TEXT NOT NULL,
  valid_from    TEXT NOT NULL,
  valid_to      TEXT NOT NULL,
  unit_cost_usd REAL NOT NULL,
  PRIMARY KEY (product_id, valid_from)
);

INSERT INTO silver.product_cost                                                          -- step 5
SELECT
  ltrim(MATNR, '0'),
  try_to_date(COST_EFF_DT, 'yyyyMMdd'),
  COALESCE(date(LEAD(try_to_date(COST_EFF_DT, 'yyyyMMdd'))
                  OVER (PARTITION BY ltrim(MATNR, '0') ORDER BY COST_EFF_DT), '-1 day'), '9999-12-31'),
  CAST(STD_COST AS REAL)
FROM bronze.cost_history_raw;
```
</details>

### ✅ Check

In [31]:
check("Products that appear more than once", n("SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM silver.product"), 0)
check("Products whose category code didn't decode", n("SELECT COUNT(*) FROM silver.product WHERE department IS NULL"), 0)
check("Overlapping cost periods", n('''SELECT COUNT(*) FROM silver.product_cost a JOIN silver.product_cost b
      ON a.product_id = b.product_id AND a.valid_from < b.valid_from AND a.valid_to >= b.valid_from'''), 0)

✅  Products that appear more than once: 0 (expected 0)
✅  Products whose category code didn't decode: 0 (expected 0)
✅  Overlapping cost periods: 0 (expected 0)


### 📌 Recap
* 162 rows (product × country) → 82 products: **one row per product**, which prevents doubled sales.
* The US copy wins because it's the master record; Canada-only products are kept.
* Codes → words; cost start dates → date ranges, so each sale gets the cost valid on its day.

---
## S5 · `silver.promotion  and  silver.promotion_item`

> 🎯 **Goal:** a readable list of promotions and which products were in each.

| | |
|---|---|
| 📥 **Input** | `bronze.promo_calendar_raw`, `bronze.promo_item_raw` |
| 📤 **Output** | `silver.promotion  and  silver.promotion_item` |
| 🧱 **One row =** | `promotion`: one promotion · `promotion_item`: one promotion × one product |
| ➡️ **Used later by** | G4 `dim_promotion`, G6 sales (promotion flag) |
| ❓ **Business questions it makes possible** | "which categories rely most on promotions?", "sales by promotion type" |

### ⚠️ What's wrong with the input

In [32]:
q("SELECT PROMO_CD, PROMO_DESC, MECH_CD, START_DT, END_DT, DISC_PCT FROM bronze.promo_calendar_raw WHERE PROMO_CD IN ('BG26W33', 'PC25W08', 'LY25W05', 'MB25W03')")

,PROMO_CD,PROMO_DESC,MECH_CD,START_DT,END_DT,DISC_PCT
0,MB25W03,Holiday Hosting - 3 for the price of 2,MULTI,2025-02-16,2025-03-01,
1,LY25W05,Snack Attack - Members save 15%,LOY,2025-03-02,2025-03-08,15
2,PC25W08,Fresh Deals - 25% off,PCT,2025-03-23,2025-04-05,25
3,BG26W33,Snack Attack - Buy one get one free,BOGO,2026-09-13,2026-09-19,


* The **mechanic** (how the promotion works) is a code: `BOGO`, `PCT`, `MULTI`, `LOY`.
* `DISC_PCT` is an **empty string** when it doesn't apply (a buy-one-get-one has no percentage). Empty text is not the
  same as "no value", and it's definitely not 0%.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `promotion_id` | `PROMO_CD` | unchanged | `BG26W33` |
| `promotion_name` | `PROMO_DESC` | unchanged | `Snack Attack - Buy one get one free` |
| `promotion_mechanic` | `MECH_CD` | decode; anything unexpected → `Other` | `BOGO` → `Buy One Get One`, `PCT` → `Percent Off`, `MULTI` → `Multi-buy`, `LOY` → `Loyalty Price` |
| `start_date, end_date` | `START_DT`, `END_DT` | text → date | `2026-09-13` → date |
| `discount_pct` | `DISC_PCT` | empty → NULL ("not applicable"); else number | `''` → `NULL`, `'25'` → `25.0` |
| `*(promotion_item)* product_id` | `MATNR` | remove leading zeros (a safety rule: this file happens to have none); keep each pair once | `20101` → `20101` |

### ▶️ Build the tables (a simple step: every change is in the column map above)

In [33]:
build("pipeline/silver/05_promotion.sql", "silver.promotion", "silver.promotion_item")
display(q("SELECT * FROM silver.promotion WHERE promotion_id IN ('BG26W33', 'PC25W08', 'LY25W05', 'MB25W03')"))
q("SELECT promotion_mechanic, COUNT(*) AS promotions, COUNT(discount_pct) AS have_a_percentage FROM silver.promotion GROUP BY 1")

▶️  built silver.promotion                            127 rows
▶️  built silver.promotion_item                       575 rows


,promotion_id,promotion_name,promotion_mechanic,start_date,end_date,discount_pct
0,BG26W33,Snack Attack - Buy one get one free,Buy One Get One,2026-09-13,2026-09-19,<NA>
1,LY25W05,Snack Attack - Members save 15%,Loyalty Price,2025-03-02,2025-03-08,15
2,MB25W03,Holiday Hosting - 3 for the price of 2,Multi-buy,2025-02-16,2025-03-01,<NA>
3,PC25W08,Fresh Deals - 25% off,Percent Off,2025-03-23,2025-04-05,25


,promotion_mechanic,promotions,have_a_percentage
0,Buy One Get One,27,0
1,Loyalty Price,24,24
2,Multi-buy,29,0
3,Percent Off,47,47


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/05_promotion.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 05 · promotion  and  promotion_item
-- Source : bronze.promo_calendar_raw, bronze.promo_item_raw
-- Grain  : promotion      -> one row per promotion
--          promotion_item -> one row per promotion per product
-- Steps  : 1. decode the mechanic: BOGO -> 'Buy One Get One', PCT -> 'Percent Off',
--             MULTI -> 'Multi-buy', LOY -> 'Loyalty Price'
--          2. type dates and the discount percentage
--          3. normalise product ids
-- =====================================================================================
DROP TABLE IF EXISTS silver.promotion;
CREATE TABLE silver.promotion (
  promotion_id       TEXT PRIMARY KEY,
  promotion_name     TEXT NOT NULL,
  promotion_mechanic TEXT NOT NULL,
  start_date         TEXT NOT NULL,
  end_date           TEXT NOT NULL,
  discount_pct       REAL
);

INSERT INTO silver.promotion
SELECT
  PROMO_CD,
  PROMO_DESC,
  CASE MECH_CD WHEN 'BOGO'  THEN 'Buy One Get One'
               WHEN 'PCT'   THEN 'Percent Off'
               WHEN 'MULTI' THEN 'Multi-buy'
               WHEN 'LOY'   THEN 'Loyalty Price'
               ELSE 'Other' END,
  try_to_date(START_DT, 'yyyy-MM-dd'),
  try_to_date(END_DT, 'yyyy-MM-dd'),
  CAST(NULLIF(DISC_PCT, '') AS REAL)
FROM bronze.promo_calendar_raw;

DROP TABLE IF EXISTS silver.promotion_item;
CREATE TABLE silver.promotion_item (
  promotion_id TEXT NOT NULL,
  product_id   TEXT NOT NULL,
  PRIMARY KEY (promotion_id, product_id)
);

INSERT OR IGNORE INTO silver.promotion_item
SELECT PROMO_CD, ltrim(MATNR, '0') FROM bronze.promo_item_raw;
```
</details>

### ✅ Check

In [34]:
check("Promotions whose mechanic didn't decode", n("SELECT COUNT(*) FROM silver.promotion WHERE promotion_mechanic = 'Other'"), 0)
check("Promotions with an unreadable date", n("SELECT COUNT(*) FROM silver.promotion WHERE start_date IS NULL OR end_date IS NULL"), 0)

✅  Promotions whose mechanic didn't decode: 0 (expected 0)
✅  Promotions with an unreadable date: 0 (expected 0)


### 📌 Recap
* Mechanic codes → words; empty strings → proper NULL; dates typed; product ids standardised.

---
## S6 · `silver.loyalty_member  and  silver.loyalty_member_restricted`

> 🎯 **Goal:** keep what's useful about loyalty members (tier, age band, home store) **without any personal data**, and lock the personal data away separately.

| | |
|---|---|
| 📥 **Input** | `bronze.crm_member_raw` |
| 📤 **Output** | `silver.loyalty_member  and  silver.loyalty_member_restricted` |
| 🧱 **One row =** | one loyalty member |
| ➡️ **Used later by** | G5 `dim_customer`; the same code is used for customers in S7 and S8 |
| ❓ **Business questions it makes possible** | "do Gold members spend more per basket than Silver?" |

### ⚠️ What's wrong with the input

In [35]:
q("SELECT CARD_NBR, EMAIL, DOB, TIER, ENROLL_DT, HOME_STR FROM bronze.crm_member_raw LIMIT 3")

,CARD_NBR,EMAIL,DOB,TIER,ENROLL_DT,HOME_STR
0,6034118822,member00001@example.com,1972-11-15,G,2019-04-02,01042
1,6046659051,member00027@example.com,1956-06-23,B,2022-05-24,00317
2,6044925192,member00028@example.com,1975-10-31,B,2022-11-05,00317


* Card number, email and date of birth are **personal data**. They must never reach the business tables or the AI.
* But we still need to **link** a member's purchases to their tier, and the tills only know the **card number**.
* Tier is a code (`G`, `S`, `B`); the home store has leading zeros.

> 💡 **Concept · pseudonymisation with a hash.** A hash function (here **SHA-256**) turns any text into a fixed
> 64-character code. The same input always gives the same code (so tables still **join**), but you can't turn the code
> back into the card number. Change one digit and the code is completely different:

In [36]:
q("SELECT '6034118822' AS card_number, sha2('6034118822', 256) AS customer_id, sha2('6034118823', 256) AS one_digit_different")

,card_number,customer_id,one_digit_different
0,6034118822,71f6e2e99597137c77eb9cd9bb4d0a74c98cd40699279b9be3f6fd92...,90e83cdc47752d26723bfb24c769d474143c52a4012aab598273f1d3...


### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `customer_id` | `CARD_NBR` | SHA-256 hash (the tills and website use the same function) | `6034118822` → `71f6e2e9...` |
| `loyalty_tier` | `TIER` | decode | `G` → `Gold`, `S` → `Silver`, `B` → `Bronze` |
| `enrollment_date` | `ENROLL_DT` | text → date | `2019-04-02` |
| `age_band` | `DOB` | age on 'today', then grouped into a band; the birth date itself is dropped | `1972-11-15` → `45-54` |
| `home_store_id` | `HOME_STR` | remove leading zeros | `01042` → `1042` |
| `extract_date` | *new* (file name) | date of the CRM extract | `crm_members_20260906.csv` → `2026-09-06` |
| `*(restricted table)* card_number, email, date_of_birth` | `CARD_NBR`, `EMAIL`, `DOB` | moved to a separate, locked-down table; **never** used downstream |  |

**Row rule:** if a card appears in several extracts, the latest extract wins.

### 🔧 Transformation 1 · Card number → pseudonymous `customer_id`
**Why:** purchases can be linked to a member's tier without anyone being able to read the card number.

### 🔧 Transformation 2 · Birth date → age band
**Why:** "spending by age group" is useful; an exact birth date is identifying and not needed.

In [37]:
q('''SELECT DOB AS "before · DOB",
            CAST((julianday('${AS_OF_DATE}') - julianday(DOB)) / 365.25 AS INTEGER) AS "age today",
            TIER AS "before · TIER", CASE TIER WHEN 'G' THEN 'Gold' WHEN 'S' THEN 'Silver' WHEN 'B' THEN 'Bronze' END AS "after · loyalty_tier"
     FROM bronze.crm_member_raw LIMIT 3''')

,before · DOB,age today,before · TIER,after · loyalty_tier
0,1972-11-15,53,G,Gold
1,1956-06-23,70,B,Bronze
2,1975-10-31,50,B,Bronze


### 🔧 Transformation 3 · Split the personal data into a restricted table
**Why:** access to one small table can be locked down (on Databricks: a column mask and permissions) while everyone
else uses the general table freely. The restricted table is **never** read by gold.

### ▶️ Build the tables

In [38]:
build("pipeline/silver/06_loyalty_member.sql", "silver.loyalty_member", "silver.loyalty_member_restricted")
print("General table (safe to use):")
display(q("SELECT * FROM silver.loyalty_member WHERE customer_id = sha2('6034118822', 256)"))
print("Restricted table (personal data, locked down):")
display(q("SELECT substr(customer_id, 1, 12) || '...' AS customer_id, card_number, email, date_of_birth FROM silver.loyalty_member_restricted WHERE customer_id = sha2('6034118822', 256)"))

▶️  built silver.loyalty_member                     1,367 rows
▶️  built silver.loyalty_member_restricted          1,367 rows
General table (safe to use):


,customer_id,loyalty_tier,enrollment_date,age_band,home_store_id,extract_date
0,71f6e2e99597137c77eb9cd9bb4d0a74c98cd40699279b9be3f6fd92...,Gold,2019-04-02,45-54,1042,2026-09-06


Restricted table (personal data, locked down):


,customer_id,card_number,email,date_of_birth
0,71f6e2e99597...,6034118822,member00001@example.com,1972-11-15


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/06_loyalty_member.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 06 · loyalty_member  and  loyalty_member_restricted
-- Source : bronze.crm_member_raw  (card number, email and date of birth = personal data)
-- Grain  : one row per loyalty member (latest extract wins)
-- Steps  : 1. replace the card number with a pseudonymous customer_id = sha2(card, 256).
--             POS and online sales use the same function, so the ids still join.
--          2. derive an age band instead of keeping the date of birth
--          3. decode tier 'G' -> 'Gold'; normalise the home store id
--          4. put the personal data in a separate RESTRICTED table.
--             On Databricks that table gets a column mask; it never flows to gold.
-- =====================================================================================
DROP TABLE IF EXISTS silver.loyalty_member;
CREATE TABLE silver.loyalty_member (
  customer_id     TEXT PRIMARY KEY,
  loyalty_tier    TEXT NOT NULL,
  enrollment_date TEXT,
  age_band        TEXT,
  home_store_id   TEXT,
  extract_date    TEXT
);

DROP TABLE IF EXISTS temp._members;
CREATE TEMP TABLE _members AS
SELECT *
FROM (
  SELECT m.*,
         try_to_date(substr(m._source_file, -12, 8), 'yyyyMMdd') AS extract_date,
         ROW_NUMBER() OVER (PARTITION BY CARD_NBR ORDER BY m._source_file DESC) AS rn
  FROM bronze.crm_member_raw m
)
WHERE rn = 1;

INSERT INTO silver.loyalty_member
SELECT
  sha2(CARD_NBR, 256),                                                              -- step 1
  CASE TIER WHEN 'G' THEN 'Gold' WHEN 'S' THEN 'Silver' WHEN 'B' THEN 'Bronze' ELSE 'Unknown' END,
  try_to_date(ENROLL_DT, 'yyyy-MM-dd'),
  CASE                                                                              -- step 2
    WHEN age < 18 THEN 'Under 18'
    WHEN age < 25 THEN '18-24'
    WHEN age < 35 THEN '25-34'
    WHEN age < 45 THEN '35-44'
    WHEN age < 55 THEN '45-54'
    WHEN age < 65 THEN '55-64'
    ELSE '65+' END,
  ltrim(HOME_STR, '0'),
  extract_date
FROM (
  SELECT *, CAST((julianday('${AS_OF_DATE}') - julianday(try_to_date(DOB, 'yyyy-MM-dd'))) / 365.25 AS INTEGER) AS age
  FROM _members
);

DROP TABLE IF EXISTS silver.loyalty_member_restricted;                              -- step 4
CREATE TABLE silver.loyalty_member_restricted (
  customer_id   TEXT PRIMARY KEY,
  card_number   TEXT NOT NULL,
  email         TEXT,
  date_of_birth TEXT
);

INSERT INTO silver.loyalty_member_restricted
SELECT sha2(CARD_NBR, 256), CARD_NBR, EMAIL, try_to_date(DOB, 'yyyy-MM-dd')
FROM _members;
```
</details>

### ✅ Check

In [39]:
cols = [r[1] for r in con.execute("PRAGMA silver.table_info(loyalty_member)")]
check("Personal-data columns in the general table", len({"card_number", "email", "date_of_birth", "CARD_NBR", "EMAIL", "DOB"} & set(cols)), 0)
check("Members lost in the split", n("SELECT COUNT(*) FROM silver.loyalty_member") - n("SELECT COUNT(*) FROM silver.loyalty_member_restricted"), 0)

✅  Personal-data columns in the general table: 0 (expected 0)
✅  Members lost in the split: 0 (expected 0)


### 📌 Recap
* Card number → irreversible `customer_id` that still joins to sales.
* Birth date → age band; tier decoded.
* Personal data isolated in a restricted table that nothing downstream uses.

---
## S7 · `silver.pos_sales_line  and  silver.quarantine_pos_sales_line`

> 🎯 **Goal:** turn the raw till log into clean, typed, US-dollar receipt lines, and set aside the rows that are broken.

| | |
|---|---|
| 📥 **Input** | `bronze.pos_tlog_raw` + `silver.fx_rate_daily` (S2) |
| 📤 **Output** | `silver.pos_sales_line  and  silver.quarantine_pos_sales_line` |
| 🧱 **One row =** | one line on a till receipt (voids and training sales **included, flagged**) |
| ➡️ **Used later by** | G5 `dim_customer`, G6 `fct_sales_line` |
| ❓ **Business questions it makes possible** | almost all of them: sales, margin, baskets, returns, promotions, loyalty |

This is the biggest step: **8 transformations**. We follow two real receipts through each one:
**receipt 88120431** (Boston, a loyalty member, 4 lines including a promotion and a **voided** line) and
**receipt 88120519** (a **return** in Toronto, in Canadian dollars).

### ⚠️ What's wrong with the input

In [40]:
q('''SELECT TRX_ID, STR_NBR, REG_NBR, LN_NBR, TRX_DT, TRX_TM, ITM_ID, QTY, EXT_AMT, DISC_AMT, TRX_TYP, LYL_CARD_NBR, PROMO_CD, CRNCY_CD
     FROM bronze.pos_tlog_raw WHERE TRX_ID IN ('88120431', '88120519') ORDER BY TRX_ID, CAST(LN_NBR AS INTEGER)''')

,TRX_ID,STR_NBR,REG_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,EXT_AMT,DISC_AMT,TRX_TYP,LYL_CARD_NBR,PROMO_CD,CRNCY_CD
0,88120431,01042,03,1,20260914,93015,000004471023,2,6.98,3.49,S,6034118822,BG26W33,USD
1,88120431,01042,03,2,20260914,93015,000000093321,1.254,1.87,0.00,S,6034118822,,USD
2,88120431,01042,03,3,20260914,93015,000004471023,1,3.49,0.00,V,6034118822,,USD
3,88120431,01042,03,4,20260914,93015,000000005012,1,3.99,0.00,S,6034118822,,USD
4,88120519,00317,01,1,20260914,94210,000000551020,1,18.99,0.00,R,,,CAD


| # | Problem | Seen above | What would go wrong if we didn't fix it |
|---|---|---|---|
| 1 | Dates and times are text; time lost its leading zero | `20260914`, `93015` (= 09:30:15) | no date maths; 09:30 might read as 93:01 |
| 2 | Ids are zero-padded | store `01042`, item `000004471023` | sales wouldn't match stores or products |
| 3 | Line type is a code; register 99 is the **training till** | `S`, `R`, `V` | voids and practice sales counted as real sales |
| 4 | **Returns have positive amounts** | Toronto `R` line: `18.99` | a return would *increase* sales |
| 5 | Loyalty card number is personal data | `6034118822` | privacy breach |
| 6 | Canadian stores sell in CAD | `CAD` | adding CAD and USD together is meaningless |
| 7 | One day was sent **twice** | (below) | that day's sales doubled |
| 8 | Some rows are broken | (below) | sales on a date that doesn't exist, no store, 9,999 units |

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `sales_line_id` | *new* | `POS-` + store + receipt + line: a unique id for the line | → `POS-1042-88120431-1` |
| `transaction_id` | `TRX_ID` | unchanged (the receipt number) | `88120431` |
| `line_number` | `LN_NBR` | text → number | `1` → `1` |
| `register_number` | `REG_NBR` | unchanged | `03` |
| `store_id` | `STR_NBR` | ① remove leading zeros | `01042` → `1042` |
| `sales_date` | `TRX_DT` | ① strict date parse (impossible dates → quarantine) | `20260914` → `2026-09-14` |
| `sales_ts` | `TRX_DT` + `TRX_TM` | ① pad time to 6 digits, combine | `93015` → `2026-09-14 09:30:15` |
| `product_id` | `ITM_ID` | ① remove leading zeros | `000004471023` → `4471023` |
| `line_type` | `TRX_TYP` | ② decode | `S` → `Sale`, `R` → `Return`, `V` → `Void` |
| `is_test_transaction` | `REG_NBR` | ② 1 if register 99 (training till) | `99` → `1` |
| `quantity_units` | `QTY` | ③ number; **× −1 for returns** | return `1` → `-1` |
| `gross_sales_amount_local` | `EXT_AMT` | ③ number; × −1 for returns | return `18.99` → `-18.99` |
| `discount_amount_local` | `DISC_AMT` | ③ number; × −1 for returns | `3.49` → `3.49` |
| `customer_id` | `LYL_CARD_NBR` | ④ SHA-256 hash; no card → `ANONYMOUS` | `6034118822` → `71f6e2e9...` |
| `fx_rate_to_usd` | *lookup* `silver.fx_rate_daily` | ⑤ that day's rate for that currency | CAD, 2026-09-14 → `0.7336` |
| `gross_sales_amount_usd, discount_amount_usd` | local amounts | ⑤ × rate, rounded to cents | `-18.99` CAD → `-13.93` USD |
| `currency_code` | `CRNCY_CD` | upper case | `CAD` |
| `unit_price_local` | `UNIT_PRC` | text → number | `3.49` |
| `promotion_code` | `PROMO_CD` | empty → NULL | `BG26W33` |
| `sales_channel` | *new* | always `In-store` (online gets `Online` in S8) | → `In-store` |
| `_source_file, _ingested_at` | audit columns | kept, for tracing and incremental loading | `pos/pos_tlog_202609.csv` |
| `*(dropped)*` | `TAX_AMT`, `TNDR_CD` | sales are reported **excluding tax**; payment method isn't needed |  |

**Row rules:** ⑥ keep one copy of any line sent twice · ⑦ rows breaking a hard rule go to the quarantine table with a reason.

### 🔧 Transformation ① · Type and standardise (dates, times, ids)
**What:** strict date parse; `printf('%06d', 93015)` = `093015` restores the missing zero; `ltrim` removes padding.
**Why:** real dates for "last week"; ids that match the store and product tables.

In [41]:
q('''SELECT TRX_DT AS "before · TRX_DT", try_to_date(TRX_DT, 'yyyyMMdd') AS "after · sales_date",
            TRX_TM AS "before · TRX_TM", printf('%06d', CAST(TRX_TM AS INTEGER)) AS "after · hhmmss",
            STR_NBR AS "before · STR_NBR", ltrim(STR_NBR, '0') AS "after · store_id",
            ITM_ID AS "before · ITM_ID", ltrim(ITM_ID, '0') AS "after · product_id"
     FROM bronze.pos_tlog_raw WHERE TRX_ID = '88120431' ''')

,before · TRX_DT,after · sales_date,before · TRX_TM,after · hhmmss,before · STR_NBR,after · store_id,before · ITM_ID,after · product_id
0,20260914,2026-09-14,93015,093015,01042,1042,000004471023,4471023
1,20260914,2026-09-14,93015,093015,01042,1042,000000093321,93321
2,20260914,2026-09-14,93015,093015,01042,1042,000004471023,4471023
3,20260914,2026-09-14,93015,093015,01042,1042,000000005012,5012


**Why a *strict* date parser?** Many tools "helpfully" roll an impossible date forward. SQLite's own `date()` turns
30 February into 2 March, which would silently move a sale into the wrong month. The strict parser returns NULL
instead, so the row is caught (transformation ⑦):

In [42]:
q("SELECT '2025-02-30' AS impossible_date, date('2025-02-30') AS lenient_parser_says, try_to_date('20250230', 'yyyyMMdd') AS strict_parser_says")

,impossible_date,lenient_parser_says,strict_parser_says
0,2025-02-30,2025-03-02,None


### 🔧 Transformation ② · Decode line type and flag the training till
**What:** `S/R/V` → `Sale/Return/Void`; register `99` → `is_test_transaction = 1`.
**Why:** a **void** is a line scanned then cancelled (not a sale); the **training till** is used to teach new staff
(fake sales). **Why flag instead of delete?** Silver keeps every real event: voids are useful for fraud and loss
analysis. Gold will exclude them from sales.

In [43]:
q('''SELECT TRX_ID, LN_NBR, TRX_TYP AS "before · TRX_TYP", CASE TRX_TYP WHEN 'S' THEN 'Sale' WHEN 'R' THEN 'Return' WHEN 'V' THEN 'Void' END AS "after · line_type",
            REG_NBR AS "before · REG_NBR", CASE WHEN REG_NBR = '99' THEN 1 ELSE 0 END AS "after · is_test_transaction"
     FROM bronze.pos_tlog_raw WHERE TRX_ID IN ('88120431', '88120519') OR rowid = (SELECT MIN(rowid) FROM bronze.pos_tlog_raw WHERE REG_NBR = '99')''')

,TRX_ID,LN_NBR,before · TRX_TYP,after · line_type,before · REG_NBR,after · is_test_transaction
0,70000129,1,S,Sale,99,1
1,88120431,1,S,Sale,03,0
2,88120431,2,S,Sale,03,0
3,88120431,3,V,Void,03,0
4,88120431,4,S,Sale,03,0
5,88120519,1,R,Return,01,0


### 🔧 Transformation ③ · Make returns negative
**What:** multiply quantity and amounts by −1 when the line is a return.
**Why:** the tills send returns as positive numbers. If we just added them up, a $18.99 refund would **add** $18.99 to
sales. With the sign carrying the meaning, a simple `SUM()` automatically nets returns off.

### 🔧 Transformation ④ · Replace the loyalty card with the pseudonymous id
**What:** the same `sha2()` as S6, or `ANONYMOUS` when no card was scanned. **Why:** privacy, while still linking to the member's tier.

### 🔧 Transformation ⑤ · Convert to US dollars
**What:** look up the rate for **that day** and **that currency** in `silver.fx_rate_daily`, and multiply.
**Why:** group reporting is in USD. Thanks to S2, weekends have a rate and USD uses 1.0, so every row uses one rule.

In [44]:
q('''SELECT p.TRX_ID, p.TRX_TYP, p.CRNCY_CD, p.EXT_AMT AS "before · EXT_AMT",
            CASE WHEN p.TRX_TYP = 'R' THEN -1 ELSE 1 END * CAST(p.EXT_AMT AS REAL) AS "③ signed",
            fx.rate_to_usd AS "⑤ rate that day",
            ROUND(CASE WHEN p.TRX_TYP = 'R' THEN -1 ELSE 1 END * CAST(p.EXT_AMT AS REAL) * fx.rate_to_usd, 2) AS "after · gross_usd",
            CASE WHEN NULLIF(TRIM(p.LYL_CARD_NBR), '') IS NULL THEN 'ANONYMOUS' ELSE substr(sha2(p.LYL_CARD_NBR, 256), 1, 12) || '...' END AS "④ after · customer_id"
     FROM bronze.pos_tlog_raw p
     LEFT JOIN silver.fx_rate_daily fx ON fx.rate_date = try_to_date(p.TRX_DT, 'yyyyMMdd') AND fx.from_currency = upper(p.CRNCY_CD)
     WHERE p.TRX_ID = '88120519' OR (p.TRX_ID = '88120431' AND p.LN_NBR = '1')''')

,TRX_ID,TRX_TYP,CRNCY_CD,before · EXT_AMT,③ signed,⑤ rate that day,after · gross_usd,④ after · customer_id
0,88120431,S,USD,6.98,6.98,1.0,6.98,71f6e2e99597...
1,88120519,R,CAD,18.99,-18.99,0.7336,-13.93,ANONYMOUS


The Toronto return: `18.99` CAD (positive) → `-18.99` CAD → `-13.93` USD.

> 💡 **Concept · natural key and deduplication.** A **natural key** is the combination of columns that identifies a
> real-world thing. A till line is identified by *store + register + receipt + line + date*. If two rows have the same
> natural key, one is a duplicate.

### 🔧 Transformation ⑥ · Remove the day that was sent twice
**What:** `ROW_NUMBER()` over the natural key; keep copy 1. **Why:** on 10 June 2025 the store system re-sent its file:

In [45]:
q('''SELECT TRX_DT AS day, COUNT(*) AS rows_received, COUNT(DISTINCT STR_NBR || REG_NBR || TRX_ID || LN_NBR) AS real_lines,
            COUNT(*) - COUNT(DISTINCT STR_NBR || REG_NBR || TRX_ID || LN_NBR) AS duplicates
     FROM bronze.pos_tlog_raw WHERE TRX_DT IN ('20250609', '20250610', '20250611') GROUP BY TRX_DT''')

,day,rows_received,real_lines,duplicates
0,20250609,155,155,0
1,20250610,200,100,100
2,20250611,133,133,0


Without this step, 10 June 2025 would show double sales.

> 💡 **Concept · quarantine.** Some rows can't be fixed automatically. Deleting them silently hides a problem at the
> source; letting them through corrupts the numbers. So we **quarantine** them: move them to a separate table **with
> the reason**, so someone can fix the source system. (On Databricks this is an *expectation* in the pipeline.)

### 🔧 Transformation ⑦ · Quarantine rows that break a hard rule
**What:** each row is tested against these rules; the first rule it breaks becomes its `failure_reason`:

| Rule | Why it's a hard rule |
|---|---|
| store number must exist | a sale must belong to a store |
| date must be a real date | a sale can't happen on 30 February |
| line type must be S, R or V | an unknown type can't be classified as a sale or not |
| item must exist | a sale must be of something |
| quantity ≤ 500 | 9,999 bags of chips on one line is a scanner error |
| an FX rate must exist | the amount can't be converted |

### ▶️ Build the tables

In [46]:
build("pipeline/silver/07_pos_sales_line.sql", "silver.pos_sales_line", "silver.quarantine_pos_sales_line")
q('''SELECT sales_line_id, store_id, sales_ts, product_id, line_type, quantity_units, currency_code,
            gross_sales_amount_local, fx_rate_to_usd, gross_sales_amount_usd, discount_amount_usd, substr(customer_id, 1, 10) AS customer_id
     FROM silver.pos_sales_line WHERE transaction_id IN ('88120431', '88120519') ORDER BY sales_line_id''')

▶️  built silver.pos_sales_line                    98,053 rows
▶️  built silver.quarantine_pos_sales_line             39 rows


,sales_line_id,store_id,sales_ts,product_id,line_type,quantity_units,currency_code,gross_sales_amount_local,fx_rate_to_usd,gross_sales_amount_usd,discount_amount_usd,customer_id
0,POS-1042-88120431-1,1042,2026-09-14 09:30:15,4471023,Sale,2.0,USD,6.98,1.0,6.98,3.49,71f6e2e995
1,POS-1042-88120431-2,1042,2026-09-14 09:30:15,93321,Sale,1.254,USD,1.87,1.0,1.87,0.0,71f6e2e995
2,POS-1042-88120431-3,1042,2026-09-14 09:30:15,4471023,Void,1.0,USD,3.49,1.0,3.49,0.0,71f6e2e995
3,POS-1042-88120431-4,1042,2026-09-14 09:30:15,5012,Sale,1.0,USD,3.99,1.0,3.99,0.0,71f6e2e995
4,POS-317-88120519-1,317,2026-09-14 09:42:10,551020,Return,-1.0,CAD,-18.99,0.7336,-13.93,0.0,ANONYMOUS


Compare with the raw rows at the top of this section: ids cleaned, date and time combined, the Toronto return is now
negative and in USD, the card number became a code. **The void (line 3) is still here, flagged `Void`**: silver keeps
every real event. And the quarantine table, with its reasons:

In [47]:
q("SELECT failure_reason, COUNT(*) AS rows, MIN(_source_file) AS example_file FROM silver.quarantine_pos_sales_line GROUP BY 1 ORDER BY 2 DESC")

,failure_reason,rows,example_file
0,missing store number,18,pos/pos_tlog_202504.csv
1,implausible quantity: 9999,9,pos/pos_tlog_202504.csv
2,invalid transaction date: 20250230,8,pos/pos_tlog_202504.csv
3,invalid transaction date: 20260230,4,pos/pos_tlog_202607.csv


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/07_pos_sales_line.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 07 · pos_sales_line  (+ quarantine_pos_sales_line)
-- Source : bronze.pos_tlog_raw   (store POS transaction log, one file per month)
-- Grain  : one row per receipt line, INCLUDING voids and training transactions
--          (silver keeps every valid source row; gold decides what counts as a sale)
-- Steps  : 1. TYPE      string dates/times -> DATE/TIMESTAMP, numbers -> REAL
--          2. NORMALISE store and item ids lose their leading zeros ('01042' -> '1042')
--          3. DECODE    TRX_TYP S/R/V -> Sale/Return/Void; REG_NBR 99 -> test transaction
--          4. SIGN      returns arrive with positive values; make them negative
--          5. PSEUDONYMISE loyalty card -> customer_id = sha2(card), or 'ANONYMOUS'
--          6. CONVERT   local currency -> USD with the daily rate from silver.fx_rate_daily
--          7. DEDUPLICATE the day the store system re-sent (same store/register/receipt/line)
--          8. VALIDATE  rows that break a hard rule go to the quarantine table with a reason
--                       (Databricks: CONSTRAINT ... EXPECT ... ON VIOLATION DROP ROW)
-- =====================================================================================
DROP TABLE IF EXISTS temp._pos_typed;
CREATE TEMP TABLE _pos_typed AS
SELECT
  NULLIF(ltrim(STR_NBR, '0'), '')                                  AS store_id,          -- 2
  TRX_ID                                                           AS transaction_id,
  CAST(LN_NBR AS INTEGER)                                          AS line_number,
  REG_NBR                                                          AS register_number,
  try_to_date(TRX_DT, 'yyyyMMdd')                                  AS sales_date,        -- 1
  printf('%06d', CAST(TRX_TM AS INTEGER))                          AS hhmmss,
  NULLIF(ltrim(ITM_ID, '0'), '')                                   AS product_id,        -- 2
  CASE TRX_TYP WHEN 'S' THEN 'Sale' WHEN 'R' THEN 'Return' WHEN 'V' THEN 'Void' END AS line_type, -- 3
  CASE WHEN REG_NBR = '99' THEN 1 ELSE 0 END                       AS is_test_transaction,
  CASE WHEN TRX_TYP = 'R' THEN -1 ELSE 1 END                       AS sign,              -- 4
  CAST(QTY AS REAL)                                                AS qty_abs,
  CAST(UNIT_PRC AS REAL)                                           AS unit_price_local,
  CAST(EXT_AMT AS REAL)                                            AS gross_abs,
  CAST(DISC_AMT AS REAL)                                           AS discount_abs,
  NULLIF(TRIM(LYL_CARD_NBR), '')                                   AS card,
  NULLIF(TRIM(PROMO_CD), '')                                       AS promotion_code,
  upper(CRNCY_CD)                                                  AS currency_code,
  TRX_DT AS raw_trx_dt, STR_NBR AS raw_str_nbr, QTY AS raw_qty,
  _source_file, _row_number, _ingested_at,
  ROW_NUMBER() OVER (PARTITION BY STR_NBR, REG_NBR, TRX_ID, LN_NBR, TRX_DT
                     ORDER BY _source_file, _row_number)          AS copy_no            -- 7
FROM bronze.pos_tlog_raw;

DROP TABLE IF EXISTS temp._pos_checked;
CREATE TEMP TABLE _pos_checked AS
SELECT t.*,
       fx.rate_to_usd AS fx_rate_to_usd,
       CASE                                                                              -- 8
         WHEN t.store_id IS NULL             THEN 'missing store number'
         WHEN t.sales_date IS NULL           THEN 'invalid transaction date: ' || t.raw_trx_dt
         WHEN t.line_type IS NULL            THEN 'unknown transaction type'
         WHEN t.product_id IS NULL           THEN 'missing item id'
         WHEN ABS(t.qty_abs) > 500           THEN 'implausible quantity: ' || t.raw_qty
         WHEN fx.rate_to_usd IS NULL         THEN 'no FX rate for ' || t.currency_code
       END AS failure_reason
FROM _pos_typed t
LEFT JOIN silver.fx_rate_daily fx                                                        -- 6
       ON fx.rate_date = t.sales_date AND fx.from_currency = t.currency_code
WHERE t.copy_no = 1;

-- Rows that fail a rule: kept, with the reason, so someone can fix the source
DROP TABLE IF EXISTS silver.quarantine_pos_sales_line;
CREATE TABLE silver.quarantine_pos_sales_line AS
SELECT failure_reason, raw_str_nbr, transaction_id, line_number, raw_trx_dt, product_id, raw_qty,
       _source_file, _row_number, _ingested_at
FROM _pos_checked
WHERE failure_reason IS NOT NULL;

DROP TABLE IF EXISTS silver.pos_sales_line;
CREATE TABLE silver.pos_sales_line (
  sales_line_id            TEXT PRIMARY KEY,
  transaction_id           TEXT    NOT NULL,
  line_number              INTEGER NOT NULL,
  register_number          TEXT,
  store_id                 TEXT    NOT NULL,
  sales_date               TEXT    NOT NULL,
  sales_ts                 TEXT    NOT NULL,
  product_id               TEXT    NOT NULL,
  customer_id              TEXT    NOT NULL,
  promotion_code           TEXT,
  sales_channel            TEXT    NOT NULL,
  line_type                TEXT    NOT NULL,
  is_test_transaction      INTEGER NOT NULL,
  quantity_units           REAL    NOT NULL,
  currency_code            TEXT    NOT NULL,
  unit_price_local         REAL,
  gross_sales_amount_local REAL    NOT NULL,
  discount_amount_local    REAL    NOT NULL,
  fx_rate_to_usd           REAL    NOT NULL,
  gross_sales_amount_usd   REAL    NOT NULL,
  discount_amount_usd      REAL    NOT NULL,
  _source_file             TEXT,
  _ingested_at             TEXT
);

INSERT INTO silver.pos_sales_line
SELECT
  'POS-' || store_id || '-' || transaction_id || '-' || line_number,
  transaction_id,
  line_number,
  register_number,
  store_id,
  sales_date,
  sales_date || ' ' || substr(hhmmss, 1, 2) || ':' || substr(hhmmss, 3, 2) || ':' || substr(hhmmss, 5, 2),
  product_id,
  CASE WHEN card IS NULL THEN 'ANONYMOUS' ELSE sha2(card, 256) END,                      -- 5
  promotion_code,
  'In-store',
  line_type,
  is_test_transaction,
  sign * qty_abs,                                                                        -- 4
  currency_code,
  unit_price_local,
  ROUND(sign * gross_abs, 2),
  ROUND(sign * discount_abs, 2),
  fx_rate_to_usd,
  ROUND(sign * gross_abs    * fx_rate_to_usd, 2),                                        -- 6
  ROUND(sign * discount_abs * fx_rate_to_usd, 2),
  _source_file,
  _ingested_at
FROM _pos_checked
WHERE failure_reason IS NULL;
```
</details>

### ✅ Check: every raw row is accounted for
Every row that arrived must end up in exactly one place: the clean table, the quarantine table, or removed as a
duplicate. This is called **reconciliation**.

In [48]:
raw_rows = n("SELECT COUNT(*) FROM bronze.pos_tlog_raw")
clean, quarantined = n("SELECT COUNT(*) FROM silver.pos_sales_line"), n("SELECT COUNT(*) FROM silver.quarantine_pos_sales_line")
duplicates = n("SELECT COUNT(*) FROM temp._pos_typed WHERE copy_no > 1")
print(f"raw {raw_rows:,}  =  clean {clean:,}  +  quarantined {quarantined}  +  duplicates {duplicates}")
check("Raw rows not accounted for", raw_rows - clean - quarantined - duplicates, 0)
check("Returns with a positive amount", n("SELECT COUNT(*) FROM silver.pos_sales_line WHERE line_type = 'Return' AND gross_sales_amount_usd > 0"), 0)
check("Card numbers left in the table", n("SELECT COUNT(*) FROM silver.pos_sales_line WHERE customer_id GLOB '60[0-9]*' AND length(customer_id) = 10"), 0)
q("SELECT line_type, is_test_transaction, COUNT(*) AS lines FROM silver.pos_sales_line GROUP BY 1, 2")

raw 98,192  =  clean 98,053  +  quarantined 39  +  duplicates 100
✅  Raw rows not accounted for: 0 (expected 0)
✅  Returns with a positive amount: 0 (expected 0)
✅  Card numbers left in the table: 0 (expected 0)


,line_type,is_test_transaction,lines
0,Return,0,426
1,Return,1,1
2,Sale,0,97032
3,Sale,1,237
4,Void,0,357


### 📌 Recap
* Typed, standardised, decoded: dates, times, ids, line types, the training till.
* Returns made negative, cards pseudonymised, everything converted to USD.
* Duplicates removed; broken rows quarantined with a reason; voids and training sales **kept but flagged**.

---
## S8 · `silver.online_sales_line`

> 🎯 **Goal:** turn online orders (nested JSON) into order lines with **exactly the same columns** as the till lines from S7.

| | |
|---|---|
| 📥 **Input** | `bronze.ecom_order_raw` + `silver.fx_rate_daily` (S2) |
| 📤 **Output** | `silver.online_sales_line` |
| 🧱 **One row =** | one line of an online order (cancelled orders **included, flagged**) |
| ➡️ **Used later by** | G5 `dim_customer`, G6 `fct_sales_line` |
| ❓ **Business questions it makes possible** | "what share of sales is online, by region?" |

### ⚠️ What's wrong with the input
Each bronze row is a whole order, stored as one JSON text:

In [49]:
order = json.loads(n("SELECT value FROM bronze.ecom_order_raw WHERE json_extract(value, '$.orderId') = 'W1007923'"))
order["lines"] = order["lines"][:2] + ["... 4 more lines ..."]
print(json.dumps(order, indent=2))

{
  "orderId": "W1007923",
  "orderTs": "2026-09-01T23:25:38-04:00",
  "fulfilmentStoreId": "9001",
  "loyaltyId": 6097499564,
  "currency": "USD",
  "status": "DELIVERED",
  "lines": [
    {
      "lineNo": 1,
      "sku": "20103",
      "qty": 1,
      "unitPrice": 4.99,
      "lineTotal": 4.99,
      "discount": 1.5,
      "promoCode": "PC26W31"
    },
    {
      "lineNo": 2,
      "sku": "4493002",
      "qty": 1,
      "unitPrice": 3.99,
      "lineTotal": 3.99,
      "discount": 0.0,
      "promoCode": null
    },
    "... 4 more lines ..."
  ]
}


* The order **lines are nested** inside the order: one row holds several products. A table needs one row per line.
* The timestamp includes a **timezone offset** (`-04:00` = New York time).
* The store id has **no** leading zeros (`9001`), unlike the tills. (Our standard "no zeros" rule handles both.)
* Some orders were `CANCELLED`.
* Field names differ from the tills (`sku` vs `ITM_ID`, `lineTotal` vs `EXT_AMT`).

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `sales_line_id` | *new* | `WEB-` + order + line | → `WEB-W1007923-1` |
| `transaction_id` | `orderId` | unchanged | `W1007923` |
| `line_number` | `lines[].lineNo` | one row per element of `lines` | `1` |
| `register_number` | *none* | NULL (no till online) |  |
| `store_id` | `fulfilmentStoreId` | remove leading zeros | `9001` |
| `sales_date` | `orderTs` | first 10 characters of the **local** time (never converted to UTC) | `2026-09-01T23:25:38-04:00` → `2026-09-01` |
| `sales_ts` | `orderTs` | local date and time without the offset | → `2026-09-01 23:25:38` |
| `product_id` | `lines[].sku` | remove leading zeros | `20103` |
| `customer_id` | `loyaltyId` | SHA-256 hash, same as the tills; none → `ANONYMOUS` |  |
| `line_type` | `status` | `CANCELLED` → `Cancelled`, otherwise `Sale` |  |
| `is_test_transaction` | *new* | always 0 |  |
| `sales_channel` | *new* | always `Online` |  |
| `quantity_units, unit_price_local` | `qty`, `unitPrice` | unchanged |  |
| `gross/discount_amount_local` | `lineTotal`, `discount` | rounded to cents | `4.99`, `1.5` |
| `fx_rate_to_usd, *_usd` | *lookup* `fx_rate_daily` | same conversion as the tills |  |

### 🔧 Transformation 1 · Flatten the nested lines
**What:** `json_each(value, '$.lines')` produces **one row per element** of the `lines` list, and `json_extract()` reads a
field. One order → many rows. (Databricks: `explode()`.) **Why:** every table needs one row per thing we count.

In [50]:
q('''SELECT json_extract(o.value, '$.orderId') AS order_id, json_extract(l.value, '$.lineNo') AS line_no,
            json_extract(l.value, '$.sku') AS sku, json_extract(l.value, '$.qty') AS qty,
            json_extract(l.value, '$.lineTotal') AS line_total, json_extract(l.value, '$.discount') AS discount
     FROM bronze.ecom_order_raw o, json_each(o.value, '$.lines') l
     WHERE json_extract(o.value, '$.orderId') = 'W1007923' ''')

,order_id,line_no,sku,qty,line_total,discount
0,W1007923,1,20103,1,4.99,1.5
1,W1007923,2,4493002,1,3.99,0.0
2,W1007923,3,5013,1,3.89,0.0
3,W1007923,4,552001,1,4.49,0.0
4,W1007923,5,4491001,2,4.98,0.0
5,W1007923,6,5102,1,5.49,0.0


### 🔧 Transformation 2 · Use the **local** date as the business date
**What:** take the date part of the local timestamp. **Why:** this order was placed at 11:25 pm in New Jersey on
**1 September**. The customer, the store and Finance all call it a 1 September sale. Converting to UTC (a common
"best practice" for timestamps) would move it to 2 September:

In [51]:
display(q('''SELECT json_extract(value, '$.orderTs') AS "before · orderTs",
                    substr(json_extract(value, '$.orderTs'), 1, 10) AS "after · sales_date (local, correct)",
                    date(json_extract(value, '$.orderTs')) AS "if converted to UTC (wrong)"
             FROM bronze.ecom_order_raw WHERE json_extract(value, '$.orderId') = 'W1007923' '''))
print("Orders that would land on the wrong day in UTC:",
      n("SELECT COUNT(*) FROM bronze.ecom_order_raw WHERE substr(json_extract(value, '$.orderTs'), 1, 10) <> date(json_extract(value, '$.orderTs'))"))

,before · orderTs,"after · sales_date (local, correct)",if converted to UTC (wrong)
0,2026-09-01T23:25:38-04:00,2026-09-01,2026-09-02


Orders that would land on the wrong day in UTC: 2066


### 🔧 Transformation 3 · Same columns as the till lines
**What:** the table is created as an exact copy of `silver.pos_sales_line`'s structure, and every field is mapped into
it (see the column map). **Why:** in gold we simply **stack** the two tables into one sales table. Different column
names or orders would make that stacking error-prone. Cancelled orders become `line_type = 'Cancelled'` (kept, flagged).

### ▶️ Build the table

In [52]:
build("pipeline/silver/08_online_sales_line.sql", "silver.online_sales_line")
q('''SELECT sales_line_id, store_id, sales_date, sales_ts, product_id, sales_channel, line_type, quantity_units,
            gross_sales_amount_local, discount_amount_local, gross_sales_amount_usd
     FROM silver.online_sales_line WHERE transaction_id = 'W1007923' ''')

▶️  built silver.online_sales_line                 49,919 rows


,sales_line_id,store_id,sales_date,sales_ts,product_id,sales_channel,line_type,quantity_units,gross_sales_amount_local,discount_amount_local,gross_sales_amount_usd
0,WEB-W1007923-1,9001,2026-09-01,2026-09-01 23:25:38,20103,Online,Sale,1,4.99,1.5,4.99
1,WEB-W1007923-2,9001,2026-09-01,2026-09-01 23:25:38,4493002,Online,Sale,1,3.99,0.0,3.99
2,WEB-W1007923-3,9001,2026-09-01,2026-09-01 23:25:38,5013,Online,Sale,1,3.89,0.0,3.89
3,WEB-W1007923-4,9001,2026-09-01,2026-09-01 23:25:38,552001,Online,Sale,1,4.49,0.0,4.49
4,WEB-W1007923-5,9001,2026-09-01,2026-09-01 23:25:38,4491001,Online,Sale,2,4.98,0.0,4.98
5,WEB-W1007923-6,9001,2026-09-01,2026-09-01 23:25:38,5102,Online,Sale,1,5.49,0.0,5.49


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/08_online_sales_line.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 08 · online_sales_line
-- Source : bronze.ecom_order_raw   (one JSON document per order, lines nested inside)
-- Grain  : one row per order line - the SAME columns as silver.pos_sales_line,
--          so gold can simply stack the two channels
-- Steps  : 1. FLATTEN   explode the nested "lines" array: one order -> many rows
--          2. BUSINESS DATE from the LOCAL timestamp. '2026-09-01T23:25:38-04:00' is a
--             1 September sale; converting to UTC first would move it to 2 September.
--          3. DECODE    status CANCELLED -> line_type 'Cancelled' (excluded in gold)
--          4. PSEUDONYMISE and CONVERT exactly like POS
-- =====================================================================================
DROP TABLE IF EXISTS silver.online_sales_line;
CREATE TABLE silver.online_sales_line AS
SELECT * FROM silver.pos_sales_line WHERE 0;          -- identical schema to the POS table

INSERT INTO silver.online_sales_line
SELECT
  'WEB-' || json_extract(o.value, '$.orderId') || '-' || json_extract(l.value, '$.lineNo'),
  json_extract(o.value, '$.orderId'),
  json_extract(l.value, '$.lineNo'),
  NULL,
  ltrim(json_extract(o.value, '$.fulfilmentStoreId'), '0'),
  substr(json_extract(o.value, '$.orderTs'), 1, 10),                                           -- 2
  replace(substr(json_extract(o.value, '$.orderTs'), 1, 19), 'T', ' '),
  ltrim(json_extract(l.value, '$.sku'), '0'),
  CASE WHEN json_extract(o.value, '$.loyaltyId') IS NULL THEN 'ANONYMOUS'
       ELSE sha2(CAST(json_extract(o.value, '$.loyaltyId') AS TEXT), 256) END,                 -- 4
  json_extract(l.value, '$.promoCode'),
  'Online',
  CASE json_extract(o.value, '$.status') WHEN 'CANCELLED' THEN 'Cancelled' ELSE 'Sale' END,  -- 3
  0,
  json_extract(l.value, '$.qty'),
  upper(json_extract(o.value, '$.currency')),
  json_extract(l.value, '$.unitPrice'),
  ROUND(json_extract(l.value, '$.lineTotal'), 2),
  ROUND(json_extract(l.value, '$.discount'), 2),
  fx.rate_to_usd,
  ROUND(json_extract(l.value, '$.lineTotal') * fx.rate_to_usd, 2),
  ROUND(json_extract(l.value, '$.discount')  * fx.rate_to_usd, 2),
  o._source_file,
  o._ingested_at
FROM bronze.ecom_order_raw o,
     json_each(o.value, '$.lines') l                                                           -- 1
JOIN silver.fx_rate_daily fx
  ON fx.rate_date = substr(json_extract(o.value, '$.orderTs'), 1, 10)
 AND fx.from_currency = upper(json_extract(o.value, '$.currency'));
```
</details>

### ✅ Check

In [53]:
pos_cols = [r[1] for r in con.execute("PRAGMA silver.table_info(pos_sales_line)")]
web_cols = [r[1] for r in con.execute("PRAGMA silver.table_info(online_sales_line)")]
check("Online and till tables have identical columns", pos_cols == web_cols, True)
check("Order lines in silver vs lines inside the JSON", n("SELECT COUNT(*) FROM silver.online_sales_line"),
      n("SELECT COUNT(*) FROM bronze.ecom_order_raw o, json_each(o.value, '$.lines')"))
q("SELECT line_type, COUNT(*) AS lines FROM silver.online_sales_line GROUP BY 1")

✅  Online and till tables have identical columns: True (expected True)
✅  Order lines in silver vs lines inside the JSON: 49919 (expected 49919)


,line_type,lines
0,Cancelled,1511
1,Sale,48408


### 📌 Recap
* Nested JSON → one row per order line.
* Business date from **local** time (2,066 orders would otherwise shift a day).
* Exactly the same columns as the till lines, so gold can stack them.

---
## S9 · `silver.inventory_daily`

> 🎯 **Goal:** clean nightly stock counts per store and product.

| | |
|---|---|
| 📥 **Input** | `bronze.inventory_raw` |
| 📤 **Output** | `silver.inventory_daily` |
| 🧱 **One row =** | one store × one product × one night |
| ➡️ **Used later by** | G7 `fct_inventory_daily` |
| ❓ **Business questions it makes possible** | "where are we out of stock?", "how much stock did we hold at the end of last week?" |

### ⚠️ What's wrong with the input

In [54]:
display(q("SELECT STR_NBR, ITM_ID, SNAP_DT, OH_QTY, ON_ORD_QTY FROM bronze.inventory_raw WHERE CAST(OH_QTY AS REAL) < 0 LIMIT 3"))
print("Negative stock readings:", n("SELECT COUNT(*) FROM bronze.inventory_raw WHERE CAST(OH_QTY AS REAL) < 0"))

,STR_NBR,ITM_ID,SNAP_DT,OH_QTY,ON_ORD_QTY
0,00322,000004471023,20260628,-1,12
1,00322,000000005012,20260629,-1,0
2,01042,000000093325,20260716,-3.000,0


Negative stock readings: 86


* Text dates and quantities; zero-padded ids (same fixes as before).
* **Negative stock** (`-3`), which is physically impossible. It happens when stock is stolen, damaged or miscounted
  without being recorded ("phantom inventory"), so the system's count drifts below zero.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `snapshot_date` | `SNAP_DT` | text → date | `20260716` → `2026-07-16` |
| `store_id` | `STR_NBR` | remove leading zeros | `01042` → `1042` |
| `product_id` | `ITM_ID` | remove leading zeros | `000000094102` → `94102` |
| `on_hand_units` | `OH_QTY` | number; **negative → 0** | `-3` → `0` |
| `on_order_units` | `ON_ORD_QTY` | number | `12` → `12` |
| `had_negative_stock` | *new* | 1 if the raw count was negative | `-3` → `1` |

**Row rule:** if a night's snapshot is sent twice, keep the latest.

### 🔧 Transformation · Clamp negative stock to zero **and flag it**
**What:** `MAX(0, qty)` plus `had_negative_stock = 1`. **Why zero?** You can't sell stock you don't have, so for
availability the real stock is zero. **Why the flag?** A negative reading signals a real problem (theft, miscounts) that
the stock team should investigate. **Never fix data silently: fix it and record that you did.**

### ▶️ Build the table

In [55]:
build("pipeline/silver/09_inventory_daily.sql", "silver.inventory_daily")
q('''SELECT i.OH_QTY AS "before · OH_QTY", s.on_hand_units AS "after · on_hand_units", s.had_negative_stock AS "after · had_negative_stock",
            s.store_id, s.product_id, s.snapshot_date
     FROM bronze.inventory_raw i JOIN silver.inventory_daily s
       ON s.store_id = ltrim(i.STR_NBR, '0') AND s.product_id = ltrim(i.ITM_ID, '0') AND s.snapshot_date = try_to_date(i.SNAP_DT, 'yyyyMMdd')
     WHERE CAST(i.OH_QTY AS REAL) < 0 LIMIT 3''')

▶️  built silver.inventory_daily                   52,052 rows


,before · OH_QTY,after · on_hand_units,after · had_negative_stock,store_id,product_id,snapshot_date
0,-1,0,1,322,4471023,2026-06-28
1,-1,0,1,322,5012,2026-06-29
2,-3.000,0,1,1042,93325,2026-07-16


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/silver/09_inventory_daily.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- SILVER 09 · inventory_daily
-- Source : bronze.inventory_raw   (nightly stock snapshot per store and item)
-- Grain  : one row per store, product and day
-- Steps  : 1. type and normalise ids and dates
--          2. negative stock is physically impossible ("phantom inventory" from
--             unrecorded shrink); clamp it to zero and FLAG it so it can be investigated
--          3. keep the latest delivery if a snapshot is re-sent
-- =====================================================================================
DROP TABLE IF EXISTS silver.inventory_daily;
CREATE TABLE silver.inventory_daily (
  snapshot_date      TEXT    NOT NULL,
  store_id           TEXT    NOT NULL,
  product_id         TEXT    NOT NULL,
  on_hand_units      REAL    NOT NULL,
  on_order_units     REAL    NOT NULL,
  had_negative_stock INTEGER NOT NULL,
  _ingested_at       TEXT,
  PRIMARY KEY (snapshot_date, store_id, product_id)
);

INSERT INTO silver.inventory_daily
SELECT snapshot_date, store_id, product_id,
       MAX(0, raw_on_hand),                                                     -- step 2
       on_order_units,
       CASE WHEN raw_on_hand < 0 THEN 1 ELSE 0 END,
       _ingested_at
FROM (
  SELECT try_to_date(SNAP_DT, 'yyyyMMdd')  AS snapshot_date,                    -- step 1
         ltrim(STR_NBR, '0')              AS store_id,
         ltrim(ITM_ID, '0')               AS product_id,
         CAST(OH_QTY AS REAL)             AS raw_on_hand,
         CAST(ON_ORD_QTY AS REAL)         AS on_order_units,
         _ingested_at,
         ROW_NUMBER() OVER (PARTITION BY SNAP_DT, STR_NBR, ITM_ID
                            ORDER BY _ingested_at DESC, _row_number DESC) AS rn  -- step 3
  FROM bronze.inventory_raw
)
WHERE rn = 1 AND snapshot_date IS NOT NULL;
```
</details>

### ✅ Check

In [56]:
check("Negative stock left", n("SELECT COUNT(*) FROM silver.inventory_daily WHERE on_hand_units < 0"), 0)
check("Negative readings flagged", n("SELECT SUM(had_negative_stock) FROM silver.inventory_daily"), 86)

✅  Negative stock left: 0 (expected 0)
✅  Negative readings flagged: 86 (expected 86)


### 📌 Recap
* Typed and standardised like every other table.
* Impossible negative stock → 0, **flagged** for investigation.

---
## Silver is done
13 tables. Two of them exist **only in silver** and never go further: the quarantine table (broken rows waiting for a
fix at the source) and the restricted member table (personal data).

In [57]:
layer_tables("silver")

,silver table,rows
0,fiscal_calendar,6573
1,fx_rate_daily,1268
2,inventory_daily,52052
3,loyalty_member,1367
4,loyalty_member_restricted,1367
5,online_sales_line,49919
6,pos_sales_line,98053
7,product,82
8,product_cost,114
9,promotion,127


---
# 6 · GOLD: organise the data the way the business thinks

Silver is clean, but it's still organised the way the **source systems** were: two separate sales tables, voids mixed
with sales, no margin, no "last week". Gold reorganises it around **business questions**.

> 💡 **Concept · the star schema: facts and dimensions.** Gold uses two kinds of table:
> * **Fact tables** record **events with numbers**: a receipt line with its amount, a nightly stock count. They're long (many rows).
> * **Dimension tables** **describe** the who / what / where / when of those events: the store, the product, the
>   customer, the promotion, the date. They hold the words people filter and group by.
>
> Each fact row points at one row in each dimension through a **key** (`store_id`, `product_id`...). Drawn out, the facts
> sit in the middle and the dimensions around them, like a star:
> ```
>                  dim_date
>                     |
>   dim_customer -- fct_sales_line -- dim_store          dim_date -- fct_inventory_daily -- dim_store
>                   /         \                                            |
>          dim_promotion    dim_product                                 dim_product
> ```

> 💡 **Concept · keys.** A **primary key** is the column whose value is unique for every row (`dim_store.store_id`). A
> **foreign key** is a column in another table that must point at an existing primary key (`fct_sales_line.store_id`).
> Locally, the database **enforces** foreign keys: a sale pointing at a store that doesn't exist stops the pipeline.

> 💡 **Concept · "Unknown" members.** What if a sale's product isn't in the product list? Leaving the key empty would
> make it vanish in joins; stopping the pipeline would block all sales. Instead each dimension has an explicit row such
> as `UNKNOWN`, `ANONYMOUS` or `NO_PROMO`. Unmatched sales point there: **no sale is ever lost** and the gap is visible.

> 💡 **Concept · upsert (MERGE).** Gold tables are **not** deleted and rebuilt. Each load does "insert the row if its
> key is new, otherwise update it" (`INSERT ... ON CONFLICT DO UPDATE` here, `MERGE` on Databricks). Running a load
> twice gives the same result (**idempotent**), and late corrections update rows in place.

**Order:** dimensions first, facts last, because facts look up dimensions.

**Contracts:** each gold table's columns, types, keys and a plain-English **comment per column** are written in a YAML
file in `contracts/gold/`. The tables are created from those files. The comments are what Genie reads to choose the
right column. Let's create the 7 empty gold tables:

In [58]:
contracts.create_gold_tables(con)
print(contracts.sqlite_ddl(contracts.load("dim_promotion")))

CREATE TABLE IF NOT EXISTS gold.dim_promotion (
  promotion_id TEXT NOT NULL,
  promotion_name TEXT NOT NULL,
  promotion_mechanic TEXT NOT NULL,
  start_date TEXT,
  end_date TEXT,
  PRIMARY KEY (promotion_id)
) STRICT;


---
## G1 · `gold.dim_date`

> 🎯 **Goal:** the calendar, plus ready-made yes/no columns for "last week", "last 4 weeks", "this quarter", "year to date" and "same period last year".

| | |
|---|---|
| 📥 **Input** | `silver.fiscal_calendar` (S1) |
| 📤 **Output** | `gold.dim_date` |
| 🧱 **One row =** | one calendar day |
| ➡️ **Used later by** | both fact tables (every date question) |
| ❓ **Business questions it makes possible** | "last week", "YTD vs last year", "this quarter" |

### ⚠️ Why silver isn't enough
"Last week" in retail means **the last complete fiscal week, Sunday to Saturday**, not "the last 7 days". "YTD vs last
year" means **the same fiscal days** last year, not the same calendar dates. Working that out from raw dates on every
question is exactly where people, and AI, go wrong. So the pipeline works it out **once** and stores the answer.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `calendar_date ... day_of_week_name (9 columns)` | `silver.fiscal_calendar` | copied unchanged |  |
| `is_last_completed_fiscal_week` | *new* | 1 if the day is in the latest fiscal week that ended before 'today' | 20–26 Sep 2026 → `1` |
| `is_last_4_completed_fiscal_weeks` | *new* | 1 if in that week or the 3 weeks before it |  |
| `is_current_fiscal_quarter` | *new* | 1 if in the same fiscal year and quarter as yesterday |  |
| `is_fiscal_ytd` | *new* | 1 if in this fiscal year, up to yesterday's fiscal day |  |
| `is_prior_fiscal_ytd` | *new* | 1 if in last fiscal year, up to the **same fiscal day** |  |

### 🔧 Transformation · Precompute the rolling flags
**What:** find "yesterday" and the last completed week relative to 'today', then compare every day against them.
**Why:** a question becomes a simple filter, `WHERE is_last_completed_fiscal_week`, that nobody can get wrong. The flags
move every day, so this table is refreshed on every run.

### ▶️ Build the table

In [59]:
build("pipeline/gold/01_load_dim_date.sql", "gold.dim_date")
q("SELECT calendar_date, day_of_week_name, fiscal_year, fiscal_week, is_last_completed_fiscal_week, is_fiscal_ytd FROM gold.dim_date WHERE is_last_completed_fiscal_week")

▶️  built gold.dim_date                             6,573 rows


,calendar_date,day_of_week_name,fiscal_year,fiscal_week,is_last_completed_fiscal_week,is_fiscal_ytd
0,2026-09-20,Sunday,2026,34,1,1
1,2026-09-21,Monday,2026,34,1,1
2,2026-09-22,Tuesday,2026,34,1,1
3,2026-09-23,Wednesday,2026,34,1,1
4,2026-09-24,Thursday,2026,34,1,1
5,2026-09-25,Friday,2026,34,1,1
6,2026-09-26,Saturday,2026,34,1,1


"Last week" is exactly Sunday 20 – Saturday 26 September 2026. And the two year-to-date periods:

In [60]:
q('''SELECT fiscal_year, MIN(calendar_date) AS first_day, MAX(calendar_date) AS last_day, COUNT(*) AS days,
            MIN(CASE WHEN fiscal_day_of_year = 1 THEN day_of_week_name END) AS starts_on
     FROM gold.dim_date WHERE is_fiscal_ytd OR is_prior_fiscal_ytd GROUP BY fiscal_year''')

,fiscal_year,first_day,last_day,days,starts_on
0,2025,2025-02-02,2025-09-27,238,Sunday
1,2026,2026-02-01,2026-09-26,238,Sunday


Same number of days, both starting on a Sunday. Last year's YTD ends on **27 September 2025**, not 26 September: same
fiscal day, different calendar date. Comparing by calendar date would compare a different mix of weekdays.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/01_load_dim_date.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 01 · dim_date
-- Built from : silver.fiscal_calendar
-- Grain      : one row per calendar day
-- Load       : upsert every run (the rolling flags move every day)
-- Steps      : 1. find "yesterday" and the last fully completed fiscal week relative to
--                 the as-of date (${AS_OF_DATE}; current_date() in production)
--              2. copy the fiscal attributes
--              3. precompute the rolling flags so "last week" or "YTD vs last year"
--                 becomes a simple filter instead of date arithmetic Genie must guess
-- =====================================================================================
WITH cal AS (
  SELECT * FROM silver.fiscal_calendar
),
yesterday AS (                                                                   -- step 1
  SELECT fiscal_year AS cur_fy, fiscal_quarter AS cur_fq, fiscal_day_of_year AS cur_fdoy
  FROM cal
  WHERE calendar_date = date('${AS_OF_DATE}', '-1 day')
),
last_week AS (
  SELECT MAX(fiscal_week_start_date) AS lw_start
  FROM cal
  WHERE date(fiscal_week_start_date, '+6 days') < '${AS_OF_DATE}'
)
INSERT INTO gold.dim_date (
  calendar_date, day_of_week_name, fiscal_year, fiscal_quarter, fiscal_period, fiscal_period_name,
  fiscal_week, fiscal_week_start_date, fiscal_day_of_year,
  is_last_completed_fiscal_week, is_last_4_completed_fiscal_weeks, is_current_fiscal_quarter,
  is_fiscal_ytd, is_prior_fiscal_ytd)
SELECT
  c.calendar_date, c.day_of_week_name, c.fiscal_year, c.fiscal_quarter, c.fiscal_period,  -- step 2
  c.fiscal_period_name, c.fiscal_week, c.fiscal_week_start_date, c.fiscal_day_of_year,
  c.fiscal_week_start_date = lw.lw_start,                                                  -- step 3
  c.fiscal_week_start_date BETWEEN date(lw.lw_start, '-21 days') AND lw.lw_start,
  c.fiscal_year = y.cur_fy AND c.fiscal_quarter = y.cur_fq,
  c.fiscal_year = y.cur_fy     AND c.fiscal_day_of_year <= y.cur_fdoy,
  c.fiscal_year = y.cur_fy - 1 AND c.fiscal_day_of_year <= y.cur_fdoy
FROM cal c
CROSS JOIN yesterday y
CROSS JOIN last_week lw
WHERE 1
ON CONFLICT (calendar_date) DO UPDATE SET
  day_of_week_name                 = excluded.day_of_week_name,
  fiscal_year                      = excluded.fiscal_year,
  fiscal_quarter                   = excluded.fiscal_quarter,
  fiscal_period                    = excluded.fiscal_period,
  fiscal_period_name               = excluded.fiscal_period_name,
  fiscal_week                      = excluded.fiscal_week,
  fiscal_week_start_date           = excluded.fiscal_week_start_date,
  fiscal_day_of_year               = excluded.fiscal_day_of_year,
  is_last_completed_fiscal_week    = excluded.is_last_completed_fiscal_week,
  is_last_4_completed_fiscal_weeks = excluded.is_last_4_completed_fiscal_weeks,
  is_current_fiscal_quarter        = excluded.is_current_fiscal_quarter,
  is_fiscal_ytd                    = excluded.is_fiscal_ytd,
  is_prior_fiscal_ytd              = excluded.is_prior_fiscal_ytd;
```
</details>

### ✅ Check

In [61]:
check("Days flagged as 'last week'", n("SELECT SUM(is_last_completed_fiscal_week) FROM gold.dim_date"), 7)
check("YTD days minus prior-YTD days", n("SELECT SUM(is_fiscal_ytd) - SUM(is_prior_fiscal_ytd) FROM gold.dim_date"), 0)

✅  Days flagged as 'last week': 7 (expected 7)
✅  YTD days minus prior-YTD days: 0 (expected 0)


### 📌 Recap
* The calendar plus 5 yes/no flags that turn tricky time questions into simple filters.

---
## G2 · `gold.dim_store`

> 🎯 **Goal:** one row per store with its **current** details, its status, and whether it counts as a *comparable* store.

| | |
|---|---|
| 📥 **Input** | `silver.store_history` (S3, current versions) + `silver.fiscal_calendar` (S1) |
| 📤 **Output** | `gold.dim_store` |
| 🧱 **One row =** | one store (+ an `UNKNOWN` row) |
| ➡️ **Used later by** | both fact tables |
| ❓ **Business questions it makes possible** | "sales by region", "like-for-like growth", "which stores opened in FY2025?" |

> 💡 **Concept · like-for-like (comparable stores).** If FreshCart opens 3 new stores, total sales grow even if every
> existing store is shrinking. **Like-for-like** growth compares only stores that traded for the **whole** of both years.
> FreshCart's rule: a store is comparable in fiscal year N if it traded **all of year N−1**:
> * opened on the **first day** of a fiscal year → comparable from the **next** year;
> * opened on **any other day** → comparable from the year **after next** (its first year was partial);
> * a **Dark Store** (online-only warehouse) → **never** comparable.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `store_id ... selling_area_sqft` | `silver.store_history` | copied from the **current** version only |  |
| `open_fiscal_year` | *lookup* `silver.fiscal_calendar` | fiscal year of the open date | `2025-06-14` → `2025` |
| `store_status` | *new* | `Closed` if closed on or before 'today', else `Open` | Buckhead → `Closed` |
| `comparable_from_fiscal_year` | *new* | the like-for-like rule above | Mississauga (opened mid-FY2025) → `2027` |
| `is_comparable_store_current_fy` | *new* | 1 if open and comparable this year |  |
| `*(dropped)* valid_from, valid_to, is_current` |  | history stays in silver; gold keeps the current picture |  |

**Extra row:** `UNKNOWN`, for any sale whose store can't be matched.

### ▶️ Build the table

In [62]:
build("pipeline/gold/02_load_dim_store.sql", "gold.dim_store")
q("SELECT store_id, store_name, store_format, region, open_date, open_fiscal_year, store_status, comparable_from_fiscal_year, is_comparable_store_current_fy FROM gold.dim_store ORDER BY store_id")

▶️  built gold.dim_store                               10 rows


,store_id,store_name,store_format,region,open_date,open_fiscal_year,store_status,comparable_from_fiscal_year,is_comparable_store_current_fy
0,1042,FreshCart Boston Seaport,Supercenter,Northeast,2016-05-14,2016,Open,2018,1
1,1107,FreshCart Hartford West,Express,Northeast,2019-10-05,2019,Open,2021,1
2,2210,FreshCart Atlanta Midtown,Neighborhood Market,Southeast,2014-08-09,2014,Open,2016,1
3,2211,FreshCart Atlanta Buckhead,Express,Southeast,2017-03-18,2017,Closed,2019,0
4,2305,FreshCart Chicago Lincoln Park,Supercenter,Midwest,2012-04-21,2012,Open,2014,1
5,3050,FreshCart Denver Highlands,Neighborhood Market,West,2024-09-07,2024,Open,2026,1
6,317,FreshCart Toronto Liberty Village,Supercenter,Canada,2018-06-02,2018,Open,2020,1
7,322,FreshCart Mississauga Square One,Express,Canada,2025-06-14,2025,Open,2027,0
8,9001,FreshCart Newark Fulfilment Center,Dark Store,Northeast,2023-03-11,2023,Open,<NA>,0
9,UNKNOWN,Unknown store,Unknown,Unknown,<NA>,<NA>,Unknown,<NA>,0


Read the last columns against FreshCart's story: **Denver** opened mid-FY2024 → comparable from 2026.
**Mississauga** opened mid-FY2025 → not until 2027. **Buckhead** closed → not comparable. **Newark** is a Dark Store →
never. The rule lives in one place; Genie only has to filter on the flag.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/02_load_dim_store.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 02 · dim_store
-- Built from : silver.store_history (current version only) + silver.fiscal_calendar
-- Grain      : one row per store, plus UNKNOWN
-- Load       : upsert every run (status and comparable flag depend on the as-of date)
-- Steps      : 1. take the CURRENT version of each store from the SCD2 history
--              2. look up the fiscal year and fiscal day on which the store opened
--              3. comparable (like-for-like) rule, written once, here:
--                   a store is comparable in fiscal year N if it traded ALL of year N-1
--                   -> opened on day 1 of a fiscal year : comparable from that year + 1
--                   -> opened on any other day          : comparable from that year + 2
--                   -> Dark Stores are never comparable
--              4. status (Open/Closed) and the current-year comparable flag
--              5. add the UNKNOWN member so no fact row is ever orphaned
-- =====================================================================================
WITH cur AS (
  SELECT fiscal_year AS cur_fy
  FROM silver.fiscal_calendar
  WHERE calendar_date = date('${AS_OF_DATE}', '-1 day')
),
s AS (
  SELECT
    h.*,
    cal.fiscal_year                                                     AS open_fiscal_year,      -- step 2
    CASE WHEN h.store_format = 'Dark Store'  THEN NULL                                            -- step 3
         WHEN cal.fiscal_day_of_year = 1     THEN cal.fiscal_year + 1
         ELSE cal.fiscal_year + 2 END                                   AS comparable_from_fiscal_year,
    CASE WHEN h.close_date IS NOT NULL AND h.close_date <= '${AS_OF_DATE}'
         THEN 'Closed' ELSE 'Open' END                                  AS store_status             -- step 4
  FROM silver.store_history h
  LEFT JOIN silver.fiscal_calendar cal ON cal.calendar_date = h.open_date
  WHERE h.is_current = 1                                                                          -- step 1
)
INSERT INTO gold.dim_store (
  store_id, store_name, store_format, city, state_province, country, region, open_date,
  open_fiscal_year, close_date, store_status, selling_area_sqft, comparable_from_fiscal_year,
  is_comparable_store_current_fy)
SELECT
  s.store_id, s.store_name, s.store_format, s.city, s.state_province, s.country, s.region,
  s.open_date, s.open_fiscal_year, s.close_date, s.store_status, s.selling_area_sqft,
  s.comparable_from_fiscal_year,
  CASE WHEN s.store_status = 'Open' AND s.comparable_from_fiscal_year <= cur.cur_fy THEN 1 ELSE 0 END
FROM s CROSS JOIN cur
WHERE 1
ON CONFLICT (store_id) DO UPDATE SET
  store_name = excluded.store_name, store_format = excluded.store_format, city = excluded.city,
  state_province = excluded.state_province, country = excluded.country, region = excluded.region,
  open_date = excluded.open_date, open_fiscal_year = excluded.open_fiscal_year,
  close_date = excluded.close_date, store_status = excluded.store_status,
  selling_area_sqft = excluded.selling_area_sqft,
  comparable_from_fiscal_year = excluded.comparable_from_fiscal_year,
  is_comparable_store_current_fy = excluded.is_comparable_store_current_fy;

INSERT INTO gold.dim_store (store_id, store_name, store_format, region, store_status,              -- step 5
                            is_comparable_store_current_fy)
VALUES ('UNKNOWN', 'Unknown store', 'Unknown', 'Unknown', 'Unknown', 0)
ON CONFLICT (store_id) DO NOTHING;
```
</details>

### ✅ Check

In [63]:
check("Stores (9 real + UNKNOWN)", n("SELECT COUNT(*) FROM gold.dim_store"), 10)
check("Comparable Dark Stores", n("SELECT COUNT(*) FROM gold.dim_store WHERE store_format = 'Dark Store' AND comparable_from_fiscal_year IS NOT NULL"), 0)

✅  Stores (9 real + UNKNOWN): 10 (expected 10)
✅  Comparable Dark Stores: 0 (expected 0)


### 📌 Recap
* Current version of each store; status and open year added.
* The like-for-like rule computed once; an `UNKNOWN` safety-net row.

---
## G3 · `gold.dim_product`

> 🎯 **Goal:** a business-friendly product list.

| | |
|---|---|
| 📥 **Input** | `silver.product` (S4) |
| 📤 **Output** | `gold.dim_product` |
| 🧱 **One row =** | one product (+ an `UNKNOWN` row) |
| ➡️ **Used later by** | both fact tables |
| ❓ **Business questions it makes possible** | "sales by department / category", "private label share" |

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `product_id, product_name, brand, department, category, subcategory, unit_of_measure, product_status` | `silver.product` | copied |  |
| `is_private_label` | *new* from `brand` | 1 if FreshCart's own brand | `FreshCart Select` → `1` |
| `*(dropped)* material_group_code, source_sales_org` |  | technical columns: noise for business users and for the AI |  |

**Extra row:** `UNKNOWN`, for items sold but missing from the product master (item `9999999` in this data).

**Why drop columns?** Every extra column is one more thing Genie might wrongly choose. Gold keeps only what answers
business questions.

### ▶️ Build the table

In [64]:
build("pipeline/gold/03_load_dim_product.sql", "gold.dim_product")
q("SELECT * FROM gold.dim_product WHERE product_id IN ('4471023', '93321', '94101', 'UNKNOWN')")

▶️  built gold.dim_product                             83 rows


,product_id,product_name,brand,department,category,subcategory,unit_of_measure,is_private_label,product_status
0,4471023,Crunchy Sea Salt Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,0,Active
1,93321,Bananas,Sunny Acres,Produce,Fresh Fruit,Bananas and Tropical,Kilogram,0,Active
2,94101,Spring Mix 142g,FreshCart,Produce,Fresh Vegetables,Salad Greens,Each,1,Active
3,UNKNOWN,Unknown product,<NA>,Unknown,Unknown,Unknown,Each,0,Unknown


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/03_load_dim_product.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 03 · dim_product
-- Built from : silver.product (already one row per SKU with decoded hierarchy)
-- Grain      : one row per product, plus UNKNOWN
-- Load       : upsert every run
-- Steps      : 1. keep only business columns (drop material_group_code, source_sales_org)
--              2. derive is_private_label from the brand
--              3. add the UNKNOWN member for items sold but missing from the master
-- =====================================================================================
INSERT INTO gold.dim_product (
  product_id, product_name, brand, department, category, subcategory, unit_of_measure,
  is_private_label, product_status)
SELECT
  product_id, product_name, brand, department, category, subcategory, unit_of_measure,   -- step 1
  CASE WHEN brand IN ('FreshCart', 'FreshCart Select') THEN 1 ELSE 0 END,              -- step 2
  product_status
FROM silver.product
WHERE 1
ON CONFLICT (product_id) DO UPDATE SET
  product_name = excluded.product_name, brand = excluded.brand, department = excluded.department,
  category = excluded.category, subcategory = excluded.subcategory,
  unit_of_measure = excluded.unit_of_measure, is_private_label = excluded.is_private_label,
  product_status = excluded.product_status;

INSERT INTO gold.dim_product (product_id, product_name, brand, department, category, subcategory,  -- step 3
                              unit_of_measure, is_private_label, product_status)
VALUES ('UNKNOWN', 'Unknown product', NULL, 'Unknown', 'Unknown', 'Unknown', 'Each', 0, 'Unknown')
ON CONFLICT (product_id) DO NOTHING;
```
</details>

### 📌 Recap
* Business columns only, a private-label flag, and an `UNKNOWN` row.

---
## G4 · `gold.dim_promotion`

> 🎯 **Goal:** the promotion list, plus a row meaning "no promotion".

| | |
|---|---|
| 📥 **Input** | `silver.promotion` (S5) |
| 📤 **Output** | `gold.dim_promotion` |
| 🧱 **One row =** | one promotion (+ `NO_PROMO`) |
| ➡️ **Used later by** | `fct_sales_line` |
| ❓ **Business questions it makes possible** | "sales by promotion mechanic" |

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `promotion_id, promotion_name, promotion_mechanic, start_date, end_date` | `silver.promotion` | copied |  |

**Extra row:** `NO_PROMO` (mechanic `No Promotion`). **Why?** Most sales are at full price. With a real row for them, "sales by mechanic" shows a *No Promotion* group instead of losing full-price sales in a join.

### ▶️ Build the table

In [65]:
build("pipeline/gold/04_load_dim_promotion.sql", "gold.dim_promotion")
q("SELECT * FROM gold.dim_promotion WHERE promotion_id IN ('NO_PROMO', 'BG26W33', 'PC25W08')")

▶️  built gold.dim_promotion                          128 rows


,promotion_id,promotion_name,promotion_mechanic,start_date,end_date
0,BG26W33,Snack Attack - Buy one get one free,Buy One Get One,2026-09-13,2026-09-19
1,NO_PROMO,No promotion,No Promotion,<NA>,<NA>
2,PC25W08,Fresh Deals - 25% off,Percent Off,2025-03-23,2025-04-05


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/04_load_dim_promotion.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 04 · dim_promotion
-- Built from : silver.promotion
-- Grain      : one row per promotion, plus NO_PROMO
-- Load       : upsert every run
-- Steps      : 1. copy the decoded promotion attributes
--              2. add NO_PROMO so full-price lines still have a promotion to join to
-- =====================================================================================
INSERT INTO gold.dim_promotion (promotion_id, promotion_name, promotion_mechanic, start_date, end_date)
SELECT promotion_id, promotion_name, promotion_mechanic, start_date, end_date
FROM silver.promotion
WHERE 1
ON CONFLICT (promotion_id) DO UPDATE SET
  promotion_name = excluded.promotion_name, promotion_mechanic = excluded.promotion_mechanic,
  start_date = excluded.start_date, end_date = excluded.end_date;

INSERT INTO gold.dim_promotion (promotion_id, promotion_name, promotion_mechanic)
VALUES ('NO_PROMO', 'No promotion', 'No Promotion')
ON CONFLICT (promotion_id) DO NOTHING;
```
</details>

### 📌 Recap
* Promotions copied, plus a `NO_PROMO` row so full-price sales are never dropped.

---
## G5 · `gold.dim_customer`

> 🎯 **Goal:** one row per loyalty customer, with **no personal data**, including customers the CRM hasn't told us about yet.

| | |
|---|---|
| 📥 **Input** | `silver.loyalty_member` (S6) + customer ids seen in `silver.pos_sales_line` and `silver.online_sales_line` |
| 📤 **Output** | `gold.dim_customer` |
| 🧱 **One row =** | one customer (+ `ANONYMOUS`) |
| ➡️ **Used later by** | `fct_sales_line` |
| ❓ **Business questions it makes possible** | "do Gold members spend more per basket than Silver?" |

> 💡 **Concept · late-arriving members.** The CRM file is dated 6 September. Someone who joined on 10 September is
> already shopping with their card, but isn't in the file yet. If the sales table required every customer to exist, their
> sales would fail. Instead we add a **placeholder** row (tier `Unknown`, `is_placeholder = 1`). When next month's CRM
> file arrives, the upsert fills in the details and sets `is_placeholder = 0`. This is why this table is **updated,
> never rebuilt**: a rebuild would delete placeholders that sales point to.

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `customer_id, loyalty_tier, age_band, home_store_id, enrollment_date` | `silver.loyalty_member` | copied (no personal data exists here) |  |
| `is_placeholder` | *new* | 0 for real members; 1 for ids seen in sales but not yet in the CRM |  |
| `*(dropped)* extract_date` |  | technical |  |

**Extra rows:** `ANONYMOUS` (tier `Anonymous`) for sales without a card, and one placeholder per late-arriving member.

### ▶️ Build the table

In [66]:
build("pipeline/gold/05_load_dim_customer.sql", "gold.dim_customer")
q("SELECT loyalty_tier, is_placeholder, COUNT(*) AS customers FROM gold.dim_customer GROUP BY 1, 2 ORDER BY 2, 3 DESC")

▶️  built gold.dim_customer                         1,373 rows


,loyalty_tier,is_placeholder,customers
0,Bronze,0,685
1,Silver,0,470
2,Gold,0,212
3,Anonymous,0,1
4,Unknown,1,5


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/05_load_dim_customer.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 05 · dim_customer
-- Built from : silver.loyalty_member  +  customer ids seen in silver sales lines
-- Grain      : one row per loyalty customer, plus ANONYMOUS
-- Load       : MERGE / upsert (not a rebuild) because of step 3
-- Steps      : 1. upsert members from the CRM extract (no personal data: pseudonymous id,
--                 tier, age band, home store)
--              2. add ANONYMOUS for baskets without a loyalty card
--              3. LATE-ARRIVING MEMBERS: a card scanned today may not be in the CRM
--                 extract until next week. Insert a placeholder (tier 'Unknown') so the
--                 sales still load; step 1 fills in the details when the CRM row arrives.
-- =====================================================================================
INSERT INTO gold.dim_customer (customer_id, loyalty_tier, age_band, home_store_id, enrollment_date,   -- step 1
                               is_placeholder)
SELECT customer_id, loyalty_tier, age_band, home_store_id, enrollment_date, 0
FROM silver.loyalty_member
WHERE 1
ON CONFLICT (customer_id) DO UPDATE SET
  loyalty_tier = excluded.loyalty_tier, age_band = excluded.age_band,
  home_store_id = excluded.home_store_id, enrollment_date = excluded.enrollment_date,
  is_placeholder = 0;

INSERT INTO gold.dim_customer (customer_id, loyalty_tier, is_placeholder)                          -- step 2
VALUES ('ANONYMOUS', 'Anonymous', 0)
ON CONFLICT (customer_id) DO NOTHING;

INSERT INTO gold.dim_customer (customer_id, loyalty_tier, is_placeholder)                          -- step 3
SELECT DISTINCT customer_id, 'Unknown', 1
FROM (SELECT customer_id FROM silver.pos_sales_line
      UNION
      SELECT customer_id FROM silver.online_sales_line)
WHERE 1
ON CONFLICT (customer_id) DO NOTHING;
```
</details>

### ✅ Check

In [67]:
print("Columns in gold.dim_customer:", [r[1] for r in con.execute("PRAGMA gold.table_info(dim_customer)")])
check("Customer ids in sales that are missing from dim_customer", n('''SELECT COUNT(*) FROM
      (SELECT customer_id FROM silver.pos_sales_line UNION SELECT customer_id FROM silver.online_sales_line)
      WHERE customer_id NOT IN (SELECT customer_id FROM gold.dim_customer)'''), 0)

Columns in gold.dim_customer: ['customer_id', 'loyalty_tier', 'age_band', 'home_store_id', 'enrollment_date', 'is_placeholder']


✅  Customer ids in sales that are missing from dim_customer: 0 (expected 0)


### 📌 Recap
* Members without personal data, plus `ANONYMOUS`, plus placeholders for members the CRM hasn't sent yet.

---
## G6 · `gold.fct_sales_line`

> 🎯 **Goal:** **the** sales table: every real sale and return, from both channels, in USD, with net sales, cost and margin calculated once.

| | |
|---|---|
| 📥 **Input** | `silver.pos_sales_line` + `silver.online_sales_line`, looked up against `dim_store`, `dim_product`, `dim_promotion` and `silver.product_cost` |
| 📤 **Output** | `gold.fct_sales_line` |
| 🧱 **One row =** | one line on a receipt or online order (only real sales and returns) |
| ➡️ **Used later by** | the semantic layer and every sales question |
| ❓ **Business questions it makes possible** | net sales, margin, baskets, returns, promotions, loyalty, online share |

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `sales_line_id, transaction_id, sales_date, sales_channel, line_type, quantity_units, currency_code` | silver | copied |  |
| `gross_sales_amount_usd, discount_amount_usd` | silver | copied | `6.98`, `3.49` |
| `store_id` | silver `store_id` | ② kept if it exists in `dim_store`, else `UNKNOWN` |  |
| `product_id` | silver `product_id` | ② kept if it exists in `dim_product`, else `UNKNOWN` | `9999999` → `UNKNOWN` |
| `promotion_id` | silver `promotion_code` | ② kept if it exists in `dim_promotion`, else `NO_PROMO` | NULL → `NO_PROMO` |
| `customer_id` | silver | copied (always exists thanks to G5) |  |
| `region` | *lookup* `dim_store` | ③ copy of the store's region, for security filters only | `Northeast` |
| `net_sales_amount_usd` | *new* | ④ gross − discount | `6.98 − 3.49 = 3.49` |
| `cost_of_goods_usd` | *lookup* `silver.product_cost` | ④ quantity × unit cost **valid on the sale date** | `2 × 2.23 = 4.46` |
| `gross_margin_usd` | *new* | ④ net sales − cost | `3.49 − 4.46 = −0.97` |
| `net_sales_amount_local` | *new* | ④ local gross − local discount |  |
| `is_promo_sale` | *new* | ⑤ 1 if a promotion matched |  |
| `is_loyalty_sale` | *new* | ⑤ 1 if not `ANONYMOUS` |  |
| `_source_ingested_at` | silver `_ingested_at` | ⑥ when the source file landed; used as the watermark |  |
| `*(dropped)*` | line_number, register_number, sales_ts, is_test_transaction, unit_price_local, fx_rate_to_usd, local gross/discount, _source_file | not needed for business questions |  |

**Row rule ①:** keep only `Sale` and `Return` lines that are not training transactions. Drop `Void` and `Cancelled`.

### 🔧 Transformation ① · Stack both channels and keep only real sales
**What:** `UNION ALL` stacks the two silver tables (possible because S8 gave them identical columns), then keep only
`Sale` and `Return`, excluding the training till. **Why:** this is the moment "every event that happened" becomes "what
the business counts as sales".

In [68]:
q('''SELECT sales_channel, line_type, is_test_transaction, COUNT(*) AS silver_rows,
            CASE WHEN line_type IN ('Sale', 'Return') AND is_test_transaction = 0 THEN '✅ kept' ELSE '❌ excluded' END AS in_gold
     FROM (SELECT * FROM silver.pos_sales_line UNION ALL SELECT * FROM silver.online_sales_line)
     GROUP BY 1, 2, 3 ORDER BY 1, 2, 3''')

,sales_channel,line_type,is_test_transaction,silver_rows,in_gold
0,In-store,Return,0,426,✅ kept
1,In-store,Return,1,1,❌ excluded
2,In-store,Sale,0,97032,✅ kept
3,In-store,Sale,1,237,❌ excluded
4,In-store,Void,0,357,❌ excluded
5,Online,Cancelled,0,1511,❌ excluded
6,Online,Sale,0,48408,✅ kept


### 🔧 Transformation ② · Point every key at a real dimension row
**What:** `LEFT JOIN` each dimension; if no match, use the Unknown member. **Why:** so the database's foreign-key check
passes and no sale disappears in a join (see Idea 2 in the SQL primer).

### 🔧 Transformation ③ · Copy the region onto each line
**Why:** on Databricks, a *row filter* lets a Northeast manager see only Northeast rows; the filter needs the region on
the fact itself. For analysis, use the region in `dim_store`.

### 🔧 Transformation ④ · Calculate the money once
**What:** net sales = gross − discount; cost = quantity × the unit cost **valid on the sale date** (a "range join":
`sales_date BETWEEN valid_from AND valid_to`); margin = net sales − cost.
**Why here?** If every report calculated margin itself, reports would disagree. Stored per line, margin simply adds up
for any store, product, week or region.

In [69]:
q('''SELECT l.sales_line_id, l.quantity_units AS qty, l.gross_sales_amount_usd AS gross, l.discount_amount_usd AS discount,
            ROUND(l.gross_sales_amount_usd - l.discount_amount_usd, 2) AS "net = gross − discount",
            c.unit_cost_usd AS "unit cost that day", c.valid_from AS "cost valid from",
            ROUND(l.quantity_units * c.unit_cost_usd, 2) AS "cost = qty × unit cost",
            ROUND(l.gross_sales_amount_usd - l.discount_amount_usd - ROUND(l.quantity_units * c.unit_cost_usd, 2), 2) AS "margin = net − cost"
     FROM silver.pos_sales_line l
     LEFT JOIN silver.product_cost c ON c.product_id = l.product_id AND l.sales_date BETWEEN c.valid_from AND c.valid_to
     WHERE l.transaction_id = '88120431' AND l.line_type = 'Sale' ''')

,sales_line_id,qty,gross,discount,net = gross − discount,unit cost that day,cost valid from,cost = qty × unit cost,margin = net − cost
0,POS-1042-88120431-1,2.0,6.98,3.49,3.49,2.23,2024-01-01,4.46,-0.97
1,POS-1042-88120431-2,1.254,1.87,0.0,1.87,0.92,2024-01-01,1.15,0.72
2,POS-1042-88120431-4,1.0,3.99,0.0,3.99,2.59,2024-01-01,2.59,1.4


Line 1 is two bags of chips on buy-one-get-one: the customer paid $3.49 for goods that cost $4.46, a **negative
margin** of −$0.97. That's real retail economics, and it only shows up because margin is calculated per line with the
cost valid on the day.

### 🔧 Transformation ⑤ · Add the promotion and loyalty flags
**Why:** "promo share of sales" and "loyalty share" become a simple sum over a yes/no column.

> 💡 **Concept · incremental load and the watermark.** Real sales tables have billions of rows; recalculating
> everything every night is slow and expensive. So each load only processes silver lines whose file arrived **after**
> the newest line already in gold. That "newest time already loaded" is the **watermark**. Combined with the upsert,
> re-running never duplicates anything.

### 🔧 Transformation ⑥ · Load only what's new (watermark)
### ▶️ Build the table

In [70]:
print("Watermark before loading:", repr(n("SELECT COALESCE(MAX(_source_ingested_at), '') FROM gold.fct_sales_line")), "(empty table: load everything)")
build("pipeline/gold/06_load_fct_sales_line.sql", "gold.fct_sales_line")
print("Watermark after loading: ", n("SELECT MAX(_source_ingested_at) FROM gold.fct_sales_line"))
q('''SELECT sales_line_id, line_type, product_id, quantity_units, net_sales_amount_usd, cost_of_goods_usd, gross_margin_usd, promotion_id, is_promo_sale, is_loyalty_sale
     FROM gold.fct_sales_line WHERE transaction_id = '88120431' ORDER BY sales_line_id''')

Watermark before loading: '' (empty table: load everything)


▶️  built gold.fct_sales_line                     145,866 rows
Watermark after loading:  2026-10-02T05:27:18+00:00


,sales_line_id,line_type,product_id,quantity_units,net_sales_amount_usd,cost_of_goods_usd,gross_margin_usd,promotion_id,is_promo_sale,is_loyalty_sale
0,POS-1042-88120431-1,Sale,4471023,2.0,3.49,4.46,-0.97,BG26W33,1,1
1,POS-1042-88120431-2,Sale,93321,1.254,1.87,1.15,0.72,NO_PROMO,0,1
2,POS-1042-88120431-4,Sale,5012,1.0,3.99,2.59,1.4,NO_PROMO,0,1


The receipt in gold: **the void (line 3) is gone**, and every line now carries net sales, cost and margin.

<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/06_load_fct_sales_line.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 06 · fct_sales_line
-- Built from : silver.pos_sales_line UNION ALL silver.online_sales_line
--              joined to gold.dim_store, gold.dim_product, gold.dim_promotion
--              and silver.product_cost
-- Grain      : one row per receipt / order line  (same grain as silver: nothing is summed)
-- Load       : INCREMENTAL. Only lines whose source file landed after the newest line
--              already in gold are processed (a "watermark"), then upserted by key,
--              so re-running never duplicates.
-- Steps      : 1. stack the two channels (identical silver schemas)
--              2. apply the business inclusion rules: keep Sale and Return,
--                 drop Void, Cancelled and training (test) transactions
--              3. resolve every key to a real dimension row or its Unknown member
--              4. copy region from dim_store (used only by row-level security)
--              5. calculate the money once:
--                   net_sales     = gross_sales - discount
--                   cost_of_goods = quantity x standard cost valid on the sale date
--                   gross_margin  = net_sales - cost_of_goods
--              6. set the is_promo_sale and is_loyalty_sale flags
-- =====================================================================================
WITH lines AS (                                                                            -- step 1
  SELECT * FROM silver.pos_sales_line
  UNION ALL
  SELECT * FROM silver.online_sales_line
),
watermark AS (
  SELECT COALESCE(MAX(_source_ingested_at), '') AS wm FROM gold.fct_sales_line
)
INSERT INTO gold.fct_sales_line (
  sales_line_id, transaction_id, sales_date, store_id, product_id, customer_id, promotion_id,
  region, sales_channel, line_type, quantity_units, gross_sales_amount_usd, discount_amount_usd,
  net_sales_amount_usd, cost_of_goods_usd, gross_margin_usd, net_sales_amount_local,
  currency_code, is_promo_sale, is_loyalty_sale, _source_ingested_at)
SELECT
  l.sales_line_id,
  l.transaction_id,
  l.sales_date,
  COALESCE(st.store_id, 'UNKNOWN'),                                                        -- step 3
  COALESCE(pr.product_id, 'UNKNOWN'),
  l.customer_id,
  COALESCE(pm.promotion_id, 'NO_PROMO'),
  COALESCE(st.region, 'Unknown'),                                                          -- step 4
  l.sales_channel,
  l.line_type,
  l.quantity_units,
  l.gross_sales_amount_usd,
  l.discount_amount_usd,
  ROUND(l.gross_sales_amount_usd - l.discount_amount_usd, 2),                              -- step 5
  ROUND(l.quantity_units * COALESCE(c.unit_cost_usd, 0), 2),
  ROUND(l.gross_sales_amount_usd - l.discount_amount_usd
        - ROUND(l.quantity_units * COALESCE(c.unit_cost_usd, 0), 2), 2),
  ROUND(l.gross_sales_amount_local - l.discount_amount_local, 2),
  l.currency_code,
  CASE WHEN pm.promotion_id IS NOT NULL THEN 1 ELSE 0 END,                                 -- step 6
  CASE WHEN l.customer_id <> 'ANONYMOUS' THEN 1 ELSE 0 END,
  l._ingested_at
FROM lines l
CROSS JOIN watermark w
LEFT JOIN gold.dim_store     st ON st.store_id     = l.store_id
LEFT JOIN gold.dim_product   pr ON pr.product_id   = l.product_id
LEFT JOIN gold.dim_promotion pm ON pm.promotion_id = l.promotion_code
LEFT JOIN silver.product_cost c ON c.product_id    = l.product_id
                               AND l.sales_date BETWEEN c.valid_from AND c.valid_to
WHERE l.line_type IN ('Sale', 'Return')                                                    -- step 2
  AND l.is_test_transaction = 0
  AND l._ingested_at > w.wm
ON CONFLICT (sales_line_id) DO UPDATE SET
  transaction_id = excluded.transaction_id, sales_date = excluded.sales_date,
  store_id = excluded.store_id, product_id = excluded.product_id,
  customer_id = excluded.customer_id, promotion_id = excluded.promotion_id,
  region = excluded.region, sales_channel = excluded.sales_channel, line_type = excluded.line_type,
  quantity_units = excluded.quantity_units, gross_sales_amount_usd = excluded.gross_sales_amount_usd,
  discount_amount_usd = excluded.discount_amount_usd, net_sales_amount_usd = excluded.net_sales_amount_usd,
  cost_of_goods_usd = excluded.cost_of_goods_usd, gross_margin_usd = excluded.gross_margin_usd,
  net_sales_amount_local = excluded.net_sales_amount_local, currency_code = excluded.currency_code,
  is_promo_sale = excluded.is_promo_sale, is_loyalty_sale = excluded.is_loyalty_sale,
  _source_ingested_at = excluded._source_ingested_at;
```
</details>

### ✅ Check

In [71]:
rows_before, sales_before = n("SELECT COUNT(*) FROM gold.fct_sales_line"), n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line")
run_sql_file(con, ROOT / "pipeline/gold/06_load_fct_sales_line.sql")          # run the same load again
check("Rows after running the load a second time", n("SELECT COUNT(*) FROM gold.fct_sales_line"), rows_before)
check("Net sales after running it a second time", n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line"), sales_before)
check("Void, cancelled or training lines in gold", n("SELECT COUNT(*) FROM gold.fct_sales_line WHERE line_type NOT IN ('Sale', 'Return')"), 0)
check("Lines where net ≠ gross − discount", n("SELECT COUNT(*) FROM gold.fct_sales_line WHERE ABS(net_sales_amount_usd - (gross_sales_amount_usd - discount_amount_usd)) > 0.011"), 0)
check("Broken foreign keys in gold", len(con.execute("PRAGMA gold.foreign_key_check").fetchall()), 0)
q('''SELECT 'product = UNKNOWN' AS unknown_member, COUNT(*) AS lines, ROUND(SUM(net_sales_amount_usd), 2) AS net_sales FROM gold.fct_sales_line WHERE product_id = 'UNKNOWN'
     UNION ALL SELECT 'promotion = NO_PROMO', COUNT(*), ROUND(SUM(net_sales_amount_usd), 2) FROM gold.fct_sales_line WHERE promotion_id = 'NO_PROMO'
     UNION ALL SELECT 'customer = ANONYMOUS', COUNT(*), ROUND(SUM(net_sales_amount_usd), 2) FROM gold.fct_sales_line WHERE customer_id = 'ANONYMOUS' ''')

✅  Rows after running the load a second time: 145866 (expected 145866)
✅  Net sales after running it a second time: 1153932.81 (expected 1153932.81)
✅  Void, cancelled or training lines in gold: 0 (expected 0)
✅  Lines where net ≠ gross − discount: 0 (expected 0)


✅  Broken foreign keys in gold: 0 (expected 0)


,unknown_member,lines,net_sales
0,product = UNKNOWN,14,90.73
1,promotion = NO_PROMO,125793,1001420.13
2,customer = ANONYMOUS,86281,684222.4


The 14 `UNKNOWN` product lines are scans of item `9999999`, which isn't in the product master: real money, kept and
visible. (No sale needed the `UNKNOWN` store, because rows without a store were quarantined in S7; the row is a safety net.)

### 📌 Recap
* Both channels stacked; only real sales and returns kept.
* Every key resolved (or pointed at an Unknown member); money calculated once with the cost valid on the day.
* Loaded incrementally with a watermark, so re-running changes nothing.

---
## G7 · `gold.fct_inventory_daily`

> 🎯 **Goal:** daily stock per store and product, valued in USD, with an out-of-stock flag.

| | |
|---|---|
| 📥 **Input** | `silver.inventory_daily` (S9), looked up against `dim_store`, `dim_product`, `silver.product_cost` |
| 📤 **Output** | `gold.fct_inventory_daily` |
| 🧱 **One row =** | one store × one product × one day |
| ➡️ **Used later by** | the semantic layer (stock questions) |
| ❓ **Business questions it makes possible** | "where are we out of stock?", "stock value at the end of last week" |

### 🗺️ Column map: where every output column comes from

| Output column | Comes from | Rule applied | Example (before → after) |
|---|---|---|---|
| `snapshot_date, on_hand_units` | silver | copied |  |
| `store_id, product_id` | silver | kept if in the dimension, else `UNKNOWN` |  |
| `region` | *lookup* `dim_store` | for security filters |  |
| `on_hand_value_usd` | *lookup* `silver.product_cost` | units × unit cost valid on that day | `22.329 kg × 0.92` |
| `is_out_of_stock` | *new* | 1 if units ≤ 0 **and** the product is `Active` |  |
| `*(dropped)* on_order_units, had_negative_stock, _ingested_at` |  | stay in silver for the stock team |  |

### 🔧 Transformation · Define "out of stock" precisely
**What:** out of stock = no stock **and** the product is active. **Why "active"?** A discontinued product running down
to zero is expected, not a problem. **What about products a store doesn't sell?** They never appear in that store's
snapshot, so they can't be counted as out of stock by mistake.

> 💡 **Concept · semi-additive numbers.** Sales can be added up across anything. **Stock can't be added across days**:
> if a shelf holds 10 units every night for a week, the stock is 10, not 70. Stock *can* be added across stores and
> products on the same day. Numbers like this are called *semi-additive*. The semantic layer (next part) handles this.

### ▶️ Build the table

In [72]:
build("pipeline/gold/07_load_fct_inventory_daily.sql", "gold.fct_inventory_daily")
q('''SELECT i.snapshot_date, s.store_name, p.product_name, i.on_hand_units, i.on_hand_value_usd, i.is_out_of_stock
     FROM gold.fct_inventory_daily i JOIN gold.dim_store s USING (store_id) JOIN gold.dim_product p USING (product_id)
     WHERE i.snapshot_date = '2026-09-26' AND s.store_id = '1042' ORDER BY i.is_out_of_stock DESC, p.product_name LIMIT 6''')

▶️  built gold.fct_inventory_daily                 52,052 rows


,snapshot_date,store_name,product_name,on_hand_units,on_hand_value_usd,is_out_of_stock
0,2026-09-26,FreshCart Boston Seaport,Aged Cheddar 400g,0.0,0.0,1
1,2026-09-26,FreshCart Boston Seaport,Bartlett Pears,0.0,0.0,1
2,2026-09-26,FreshCart Boston Seaport,Ground Beef 85% 500g,0.0,0.0,1
3,2026-09-26,FreshCart Boston Seaport,Spring Mix 142g,0.0,0.0,1
4,2026-09-26,FreshCart Boston Seaport,2% Milk 4L,17.0,43.01,0
5,2026-09-26,FreshCart Boston Seaport,Atlantic Salmon Fillet,3.827,67.89,0


<details>
<summary><b>📄 Show the full SQL file that was just run: <code>pipeline/gold/07_load_fct_inventory_daily.sql</code></b> (optional; every step above is one part of it)</summary>

```sql
-- =====================================================================================
-- GOLD 07 · fct_inventory_daily
-- Built from : silver.inventory_daily joined to gold.dim_store, gold.dim_product
--              and silver.product_cost
-- Grain      : one row per store, ranged product and day
-- Load       : upsert by (snapshot_date, store_id, product_id); re-sending a day replaces it
-- Steps      : 1. resolve keys to dimension rows (or UNKNOWN)
--              2. copy region for row-level security
--              3. value the stock at the standard cost valid on the snapshot date
--              4. out-of-stock = an ACTIVE product with no sellable stock.
--                 Products a store does not range never appear in the snapshot, so they
--                 can never be counted as out of stock by mistake.
-- =====================================================================================
INSERT INTO gold.fct_inventory_daily (snapshot_date, store_id, product_id, region, on_hand_units,
                                      on_hand_value_usd, is_out_of_stock)
SELECT
  i.snapshot_date,
  COALESCE(st.store_id, 'UNKNOWN'),                                                        -- step 1
  COALESCE(pr.product_id, 'UNKNOWN'),
  COALESCE(st.region, 'Unknown'),                                                          -- step 2
  i.on_hand_units,
  ROUND(i.on_hand_units * COALESCE(c.unit_cost_usd, 0), 2),                                -- step 3
  CASE WHEN i.on_hand_units <= 0 AND pr.product_status = 'Active' THEN 1 ELSE 0 END        -- step 4
FROM silver.inventory_daily i
LEFT JOIN gold.dim_store   st ON st.store_id   = i.store_id
LEFT JOIN gold.dim_product pr ON pr.product_id = i.product_id
LEFT JOIN silver.product_cost c ON c.product_id = i.product_id
                               AND i.snapshot_date BETWEEN c.valid_from AND c.valid_to
WHERE 1
ON CONFLICT (snapshot_date, store_id, product_id) DO UPDATE SET
  region = excluded.region, on_hand_units = excluded.on_hand_units,
  on_hand_value_usd = excluded.on_hand_value_usd, is_out_of_stock = excluded.is_out_of_stock;
```
</details>

### ✅ Check

In [73]:
check("Stock rows for an UNKNOWN store or product", n("SELECT COUNT(*) FROM gold.fct_inventory_daily WHERE store_id = 'UNKNOWN' OR product_id = 'UNKNOWN'"), 0)
check("Discontinued products flagged out of stock", n('''SELECT COUNT(*) FROM gold.fct_inventory_daily i JOIN gold.dim_product p USING (product_id)
      WHERE i.is_out_of_stock = 1 AND p.product_status <> 'Active' '''), 0)

✅  Stock rows for an UNKNOWN store or product: 0 (expected 0)
✅  Discontinued products flagged out of stock: 0 (expected 0)


### 📌 Recap
* Stock valued at the cost valid on the day; out-of-stock defined once, only for active products.

---
## Gold is done

In [74]:
display(layer_tables("gold"))
check("Broken foreign keys anywhere in gold", len(con.execute("PRAGMA gold.foreign_key_check").fetchall()), 0)

,gold table,rows
0,_column_comments,76
1,_table_comments,7
2,dim_customer,1373
3,dim_date,6573
4,dim_product,83
5,dim_promotion,128
6,dim_store,10
7,fct_inventory_daily,52052
8,fct_sales_line,145866


✅  Broken foreign keys anywhere in gold: 0 (expected 0)


---
# 7 · SEMANTIC LAYER: define every KPI once

Even with perfect tables, a KPI can still be calculated wrongly. Two classic mistakes, shown on our real data:

### Mistake 1 · Averaging percentages
Margin % must be **total margin ÷ total net sales** ("ratio of sums"). Averaging each line's margin % gives a $1 line
the same weight as a $100 line:

In [75]:
q('''SELECT p.department,
            ROUND(SUM(f.gross_margin_usd) / SUM(f.net_sales_amount_usd), 4)       AS "✅ right: SUM(margin) / SUM(sales)",
            ROUND(AVG(f.gross_margin_usd / NULLIF(f.net_sales_amount_usd, 0)), 4) AS "❌ wrong: AVG(line margin %)"
     FROM gold.fct_sales_line f JOIN gold.dim_product p USING (product_id) JOIN gold.dim_date d ON d.calendar_date = f.sales_date
     WHERE d.is_fiscal_ytd AND p.department <> 'Unknown' GROUP BY 1 ORDER BY 1''')

,department,✅ right: SUM(margin) / SUM(sales),❌ wrong: AVG(line margin %)
0,Bakery,0.4979,0.4951
1,Dairy and Eggs,0.2377,0.2425
2,Deli,0.4378,0.4294
3,Grocery,0.3259,0.3253
4,Household,0.304,0.3099
5,Meat and Seafood,0.2508,0.2547
6,Produce,0.3563,0.3569


### Mistake 2 · Adding stock across days
"How much Produce stock did we hold at the end of last week?" Adding up 7 nightly counts gives about 7 times the real stock:

In [76]:
q('''SELECT ROUND(SUM(i.on_hand_units), 1) AS "❌ wrong: sum of 7 nights",
            ROUND(SUM(i.on_hand_units) FILTER (WHERE i.snapshot_date = (SELECT MAX(calendar_date) FROM gold.dim_date WHERE is_last_completed_fiscal_week)), 1) AS "✅ right: last night of the week"
     FROM gold.fct_inventory_daily i JOIN gold.dim_product p USING (product_id) JOIN gold.dim_date d ON d.calendar_date = i.snapshot_date
     WHERE d.is_last_completed_fiscal_week AND p.department = 'Produce' ''')

,❌ wrong: sum of 7 nights,✅ right: last night of the week
0,14002.7,1903.1


### The fix: metric definitions
The **semantic layer** writes each KPI down once, in YAML (`semantic/sales_metrics.yaml`, `semantic/inventory_metrics.yaml`),
with its meaning and the words people use for it. On Databricks these become *metric views*; here, `freshcart/metrics.py`
reads the same YAML. Everyone, including Genie, then asks for `MEASURE(gross_margin_pct)` and gets the right answer.

In [77]:
import yaml
spec = yaml.safe_load((ROOT / "semantic/sales_metrics.yaml").read_text())
pd.DataFrame([(m["name"], m["expr"], ", ".join(m.get("synonyms", [])[:4])) for m in spec["measures"]],
             columns=["measure", "defined once as", "people also call it"])

,measure,defined once as,people also call it
0,net_sales,SUM(net_sales_amount_usd),"sales, revenue, turnover, net revenue"
1,gross_sales,SUM(gross_sales_amount_usd),"gross sales, sales before discount"
2,discount_amount,SUM(discount_amount_usd),"discounts, promotional discount, promo spend"
3,units_sold,SUM(quantity_units),"units, volume, quantity, items sold"
4,transactions,COUNT(DISTINCT transaction_id) FILTER (WHERE line_type =...,"baskets, receipts, orders, transaction count"
5,average_basket_value,"MEASURE(net_sales) / NULLIF(MEASURE(transactions), 0)","basket size, basket value, AOV, average order value"
6,units_per_transaction,"MEASURE(units_sold) / NULLIF(MEASURE(transactions), 0)","UPT, items per basket"
7,gross_margin,SUM(gross_margin_usd),"margin, GM, gross profit"
8,gross_margin_pct,"MEASURE(gross_margin) / NULLIF(MEASURE(net_sales), 0)","margin %, GM%, margin rate, margin percentage"
9,sales_before_returns,SUM(net_sales_amount_usd) FILTER (WHERE line_type = 'Sale'),


### Question 1 from the business: *"What were net sales and margin by region last week?"*

In [78]:
from freshcart import benchmarks          # reads the Genie space from the bundle (genie_bundle/), its only definition
example = next(e for e in benchmarks.load_examples() if e["title"] == "What were net sales and margin by region last week?")
sql = benchmarks.local_sql(example["sql"])
print(sql)
cols, rows = metrics.run(con, sql)
pd.DataFrame(rows, columns=cols)

SELECT region,
       MEASURE(net_sales)        AS net_sales,
       MEASURE(gross_margin)     AS gross_margin,
       MEASURE(gross_margin_pct) AS gross_margin_pct
FROM   freshcart.semantic.sales_metrics
WHERE  is_last_completed_fiscal_week
GROUP BY ALL
ORDER BY net_sales DESC


,region,net_sales,gross_margin,gross_margin_pct
0,Northeast,5579.91,2003.78,0.359106
1,Canada,2706.50,1020.47,0.377044
2,Midwest,2543.06,911.63,0.358478
3,Southeast,1065.82,388.83,0.364818
4,West,939.99,328.42,0.349387


Short, readable, and impossible to get wrong: "last week" is a flag from G1, margin % is a ratio of sums from the
metric definition, region comes from `dim_store`. Stock uses the "last day" rule automatically:

In [79]:
cols, rows = metrics.run(con, "SELECT MEASURE(on_hand_units) AS produce_stock_units FROM freshcart.semantic.inventory_metrics WHERE is_last_completed_fiscal_week AND department = 'Produce'")
pd.DataFrame(rows, columns=cols)

,produce_stock_units
0,1903.059


### A trusted function for like-for-like growth
High-stakes calculations get a **trusted function** instead of trusting anyone (or any AI) to write the SQL. It uses the
comparable-store flag from G2 and the same fiscal days from G1:

In [80]:
cols, rows = metrics.run(con, "SELECT * FROM freshcart.semantic.fn_like_for_like_sales(2026)")
pd.DataFrame(rows, columns=cols)

,region,comparable_stores,net_sales_ty,net_sales_ly,lfl_growth_pct
0,Canada,1,79792.37,79063.36,0.009221
1,Midwest,1,84177.52,83642.02,0.006402
2,Northeast,2,81529.03,62338.23,0.307850
3,Southeast,1,40586.99,40809.28,-0.005447
4,West,1,39419.21,38917.49,0.012892


Northeast grows ~31% like-for-like. Why? It has two comparable stores, and Boston Seaport was **remodelled into a
bigger Supercenter** on 1 Feb 2026 (the SCD2 change we tracked in S3):

In [81]:
q('''SELECT s.store_name, s.store_format,
            ROUND(SUM(f.net_sales_amount_usd) FILTER (WHERE d.is_fiscal_ytd), 2)       AS net_sales_ytd,
            ROUND(SUM(f.net_sales_amount_usd) FILTER (WHERE d.is_prior_fiscal_ytd), 2) AS net_sales_prior_ytd,
            ROUND(SUM(f.net_sales_amount_usd) FILTER (WHERE d.is_fiscal_ytd)
                  / SUM(f.net_sales_amount_usd) FILTER (WHERE d.is_prior_fiscal_ytd) - 1, 3) AS growth
     FROM gold.fct_sales_line f JOIN gold.dim_date d ON d.calendar_date = f.sales_date JOIN gold.dim_store s USING (store_id)
     WHERE s.region = 'Northeast' AND s.is_comparable_store_current_fy = 1 GROUP BY 1, 2''')

,store_name,store_format,net_sales_ytd,net_sales_prior_ytd,growth
0,FreshCart Boston Seaport,Supercenter,57509.28,38574.73,0.491
1,FreshCart Hartford West,Express,24019.75,23763.5,0.011


---
# 8 · Did it work? Three proofs

### Proof 1 · All data-quality rules pass

In [82]:
for layer, rule, violations in quality.run_rules(con):
    check(f"[{layer}] {rule}", violations, 0)

✅  [gold] No voided, cancelled or training lines in the sales fact: 0 (expected 0)
✅  [gold] Returns are negative, sales are positive: 0 (expected 0)
✅  [gold] net_sales = gross_sales - discount on every line: 0 (expected 0)
✅  [gold] gross_margin = net_sales - cost_of_goods on every line: 0 (expected 0)
✅  [gold] Every sales date is in dim_date: 0 (expected 0)
✅  [gold] Exactly 7 days flagged as last completed fiscal week: 0 (expected 0)
✅  [gold] YTD and prior YTD cover the same number of days: 0 (expected 0)
✅  [gold] No stock row for an UNKNOWN store or product: 0 (expected 0)
✅  [silver] One current version per store in the SCD2 history: 0 (expected 0)
✅  [silver] An FX rate exists for every day up to the as-of date: 0 (expected 0)
✅  [silver] One product row per SKU after deduplication: 0 (expected 0)
✅  [silver] No personal data outside the restricted table: 0 (expected 0)


### Proof 2 · An independent recount from the raw files
The strongest test: recalculate total net sales **straight from the raw files in plain Python**, without any of the SQL
above, applying the same business rules. If two completely separate implementations agree to the cent, the pipeline
is doing what we think it does.

In [83]:
with open(C.RAW_DIR / "finance/fx_rates.csv", newline="") as fh:
    published = {date.fromisoformat(r["RATE_DT"]): float(r["RATE"]) for r in csv.DictReader(fh)}
fx, day, last = {}, min(published), None
while day <= C.AS_OF_DATE:                                           # forward-fill weekend rates
    last = published.get(day, last); fx[day] = last; day = date.fromordinal(day.toordinal() + 1)
rate = lambda ccy, d: 1.0 if ccy == "USD" else fx[d]

total, seen = 0.0, set()
for path in sorted((C.RAW_DIR / "pos").glob("*.csv")):
    for r in csv.DictReader(open(path, newline="")):
        key = (r["STR_NBR"], r["REG_NBR"], r["TRX_ID"], r["LN_NBR"], r["TRX_DT"])
        if key in seen: continue                                    # duplicate
        seen.add(key)
        try: d = date(int(r["TRX_DT"][:4]), int(r["TRX_DT"][4:6]), int(r["TRX_DT"][6:]))
        except ValueError: continue                                  # impossible date
        if not r["STR_NBR"] or abs(float(r["QTY"])) > 500: continue # quarantined
        if r["TRX_TYP"] == "V" or r["REG_NBR"] == "99": continue    # void or training
        sign, k = (-1 if r["TRX_TYP"] == "R" else 1), rate(r["CRNCY_CD"], d)
        total += round(round(sign * float(r["EXT_AMT"]) * k, 2) - round(sign * float(r["DISC_AMT"]) * k, 2), 2)
for path in sorted((C.RAW_DIR / "ecommerce").glob("*.ndjson")):
    for line in open(path):
        o = json.loads(line)
        if o["status"] == "CANCELLED": continue
        k = rate(o["currency"], date.fromisoformat(o["orderTs"][:10]))
        total += sum(round(round(l["lineTotal"] * k, 2) - round(l["discount"] * k, 2), 2) for l in o["lines"])

gold_total = n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line")
print(f"Recounted from raw files in Python : {total:,.2f} USD")
print(f"In gold.fct_sales_line             : {gold_total:,.2f} USD")
check("Difference in cents", round((gold_total - total) * 100), 0)

Recounted from raw files in Python : 1,153,932.81 USD
In gold.fct_sales_line             : 1,153,932.81 USD
✅  Difference in cents: 0 (expected 0)


### Proof 3 · Follow one receipt line through every layer
Receipt 88120431, line 2: 1.254 kg of bananas.

In [84]:
with open(C.RAW_DIR / "pos/pos_tlog_202609.csv", encoding="utf-8") as fh:
    fh.readline(); raw_line = next(l.strip() for l in fh if l.startswith("88120431,01042,03,2,"))
print("RAW FILE  pos/pos_tlog_202609.csv:", raw_line)
print("\nBRONZE  (same text, plus where it came from)")
display(q("SELECT STR_NBR, TRX_DT, TRX_TM, ITM_ID, QTY, EXT_AMT, TRX_TYP, _source_file, _row_number FROM bronze.pos_tlog_raw WHERE TRX_ID = '88120431' AND LN_NBR = '2'"))
print("SILVER  (typed, standardised, converted)")
display(q("SELECT sales_line_id, store_id, sales_ts, product_id, line_type, quantity_units, gross_sales_amount_usd FROM silver.pos_sales_line WHERE sales_line_id = 'POS-1042-88120431-2'"))
print("GOLD    (a real sale, with cost and margin, described by its dimensions)")
display(q('''SELECT d.fiscal_year, d.fiscal_week, s.store_name, s.region, p.product_name, p.department,
                    f.quantity_units, f.net_sales_amount_usd, f.cost_of_goods_usd, f.gross_margin_usd
             FROM gold.fct_sales_line f JOIN gold.dim_date d ON d.calendar_date = f.sales_date
             JOIN gold.dim_store s USING (store_id) JOIN gold.dim_product p USING (product_id)
             WHERE f.sales_line_id = 'POS-1042-88120431-2' '''))

RAW FILE  pos/pos_tlog_202609.csv: 88120431,01042,03,2,20260914,93015,000000093321,1.254,1.49,1.87,0.00,0.00,S,6034118822,,USD,CC

BRONZE  (same text, plus where it came from)


,STR_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,EXT_AMT,TRX_TYP,_source_file,_row_number
0,01042,20260914,93015,000000093321,1.254,1.87,S,pos/pos_tlog_202609.csv,2162


SILVER  (typed, standardised, converted)


,sales_line_id,store_id,sales_ts,product_id,line_type,quantity_units,gross_sales_amount_usd
0,POS-1042-88120431-2,1042,2026-09-14 09:30:15,93321,Sale,1.254,1.87


GOLD    (a real sale, with cost and margin, described by its dimensions)


,fiscal_year,fiscal_week,store_name,region,product_name,department,quantity_units,net_sales_amount_usd,cost_of_goods_usd,gross_margin_usd
0,2026,33,FreshCart Boston Seaport,Northeast,Bananas,Produce,1.254,1.87,1.15,0.72


---
# 9 · Summary

### Every transformation, in one table

| Problem in the raw data | Fix | Why | Step |
|---|---|---|---|
| Dates, times and numbers stored as text, in 3 different date formats | strict parsing with the format stated | date maths; no silent day/month swaps; impossible dates caught | S1, S3, S7, S9 |
| Same store/product written differently (`01042`, `1042`) | remove leading zeros everywhere | tables must match | S3, S4, S7, S8, S9 |
| Codes (`NM`, `02`, `C0412`, `G`, `BOGO`, `S/R/V`) | decode with lookups / `CASE` | people and AI ask in words | S3, S4, S5, S6, S7 |
| Labels like `FY2026`, `P01` | turn into numbers | "last year" = year − 1 | S1 |
| Exchange rates only on weekdays | forward fill + flag; USD = 1.0 | every sale can be converted | S2 |
| Store details in repeated snapshots | keep only changes, with date ranges (SCD2) | history without duplicates | S3 |
| Every product listed twice | one row per product, prefer US | **prevents doubled sales** | S4 |
| Cost only has a start date | date ranges | the right cost for each sale date | S4 |
| Card numbers, emails, birthdays | hash, age band, restricted table | privacy; still joinable | S6 |
| Returns with positive amounts | make negative | refunds reduce sales | S7 |
| Voids, training till, cancelled orders | keep and flag in silver, exclude in gold | keep evidence, count only real sales | S7, S8, G6 |
| A day sent twice | deduplicate on the natural key | no double counting | S7 |
| Broken rows | quarantine with a reason | never silently drop or let through | S7 |
| CAD amounts | convert at that day's rate | one currency for reporting | S7, S8 |
| Nested JSON, timezone offsets | flatten; local business date | one row per line; the right day | S8 |
| Two sales tables with different shapes | identical columns, then stacked | one sales table | S8, G6 |
| Negative stock | clamp to zero + flag | realistic availability; problem visible | S9 |
| "Last week", "YTD" need a retail calendar | precomputed flags | simple, correct filters | G1 |
| Like-for-like needs a rule | computed once per store | consistent growth figures | G2 |
| Keys that don't match | Unknown members | no sale lost in a join | G2–G6 |
| Members not in the CRM yet | placeholder customers | sales load anyway; filled in later | G5 |
| Margin calculated differently everywhere | net, cost, margin stored per line | every report agrees | G6 |
| Reprocessing everything is slow | watermark + upsert | fast, safe re-runs | G6 |
| Percentages averaged; stock summed over days | metric definitions | KPIs always calculated correctly | Semantic |

### Glossary
| Term | Meaning |
|---|---|
| **Bronze / Silver / Gold** | raw copy / cleaned / organised for the business |
| **Grain** | what one row of a table means |
| **Primary key / foreign key** | the unique id of a row / a column pointing at another table's id |
| **Natural key** | the columns that identify a real-world thing in the source (store + register + receipt + line + date) |
| **Deduplicate** | keep one row per natural key |
| **Fan-out** | a join that multiplies rows because the other side isn't unique |
| **Forward fill** | fill a gap with the last known value |
| **SCD2** | keep one row per version of a record, with valid-from/valid-to dates |
| **Quarantine** | a table of rejected rows with the reason |
| **Pseudonymise** | replace an identifier with a code that still joins but can't be reversed |
| **Fact / dimension** | table of events with numbers / table describing who-what-where-when |
| **Unknown member** | a dimension row that catches unmatched keys |
| **Upsert / MERGE** | insert if new, update if the key exists |
| **Idempotent** | running twice gives the same result as running once |
| **Incremental load / watermark** | process only new data / the "latest already loaded" marker |
| **Semi-additive** | can be summed across some things (stores) but not others (days) |
| **Ratio of sums** | compute a % as total ÷ total, never as an average of %s |
| **Semantic layer / metric view** | KPIs defined once, in one place |

### Where to go next
* The same design on Databricks: the `databricks/` folder (`read_files` for bronze, a Lakeflow pipeline for silver,
  `MERGE` for gold, metric views, row filters and column masks).
* The written docs in `docs/`, one page per layer.
* Experiment: change `AS_OF_DATE` in `freshcart/config.py`, re-run this notebook, and watch "last week" and the
  comparable stores move.